<!-- derived-from: ../../02_%E7%BB%BC%E5%90%88%E5%AE%9E%E6%88%98/01_%E7%BB%BC%E5%90%88%E5%AE%9E%E6%88%98-LangChain-%E5%9F%BA%E7%A1%80%E7%89%88.ipynb -->
> English mirror, derived from the Chinese master: [01_langchain-baseline.ipynb](../../02_%E7%BB%BC%E5%90%88%E5%AE%9E%E6%88%98/01_%E7%BB%BC%E5%90%88%E5%AE%9E%E6%88%98-LangChain-%E5%9F%BA%E7%A1%80%E7%89%88.ipynb). The Chinese notebook is the source of truth.
>
> Cell outputs are intentionally empty in this mirror: they hold Chinese corpus text, and this course never hand-writes model output. Run the notebook on the vLLM + Xinference + Milvus stack to populate them.
> **Runtime preparation**: before running anything else, do two basic setup steps.
> 1. `warnings.filterwarnings("ignore")`: silence runtime warnings so deprecation warnings from legacy LangChain APIs (such as `langchain_classic`) do not distract from the lesson.
> 2. Clear the `HTTP_PROXY` / `HTTPS_PROXY` / `ALL_PROXY` environment variables: prevent proxy settings on the host or inside the container from breaking access to vLLM, Xinference, Milvus and the other services.


In [ ]:
import warnings
warnings.filterwarnings("ignore")


import os
for k in [
    "HTTP_PROXY",
    "HTTPS_PROXY",
    "ALL_PROXY",
    "http_proxy",
    "https_proxy",
    "all_proxy"
]:
    os.environ.pop(k, None)

# Introduction to Retrieval-Augmented Generation (RAG)

## What RAG is

Large language models (LLMs) are far more capable than traditional language models, yet in some situations they still fail to give accurate answers. To address the challenges LLMs face when generating text, and to improve model performance and output quality, researchers proposed a new architecture: **Retrieval-Augmented Generation (RAG)**. This architecture **retrieves relevant information from a large knowledge base and feeds it to the language model as grounding context, so the model produces far more accurate answers**, which substantially improves both correctness and depth.

The main problems current LLMs still have:

- **Factual drift / hallucination:** an LLM sometimes produces information that contradicts objective facts, so users receive inaccurate content. RAG retrieves from a data source to steer generation, which keeps the output precise and trustworthy and reduces factual drift.

- **Stale knowledge:** an LLM is trained on a static dataset, so its knowledge lags behind events. RAG retrieves the freshest data on demand, keeping content current and accurate.

- **No provenance:** generated text usually has no explicit source, which hurts credibility. RAG links the answer back to the retrieved original material, improving traceability and user trust.

- **Weak domain expertise:** an LLM is often mediocre on specialist domains, which lowers answer quality there. RAG retrieves domain documents and hands the model rich context, improving both quality and depth of specialist answers.

- **Limited reasoning:** on complex questions an LLM may lack the reasoning it needs, which hurts comprehension and answers. RAG combines retrieved evidence with generation, supplying extra background and data that strengthen reasoning and understanding.

- **Narrow scenario fit:** an LLM must stay accurate across many scenarios, but a single model rarely fits them all. RAG lets the LLM pull scenario-specific data, adapting flexibly to Q&A systems, recommender systems and more.

- **Weak long-context handling:** an LLM is bounded by its context window and must process content sequentially, so longer input means slower processing. RAG retrieves and assembles the relevant parts of long material, breaking the input-length limit, cutting cost and raising overall throughput.

## How RAG works

RAG is a complete system whose workflow splits into four stages: data processing, retrieval, augmentation and generation.

<img src="../../assets/rag-10.png" width="100%">

1. **Data processing stage**
   1. Clean and process the raw data.
   2. Convert the cleaned data into a format the retrieval model can consume.
   3. Store the processed data in the appropriate database.
2. **Retrieval stage**
   1. Feed the user question into the retrieval system and pull relevant information from the database.
3. **Augmentation stage**
   1. Process and augment the retrieved information so the generation model can understand and use it well.
4. **Generation stage**
   1. Feed the augmented information into the generation model, which produces the answer from it.

## RAG vs fine-tuning

When it comes to improving an LLM, RAG and fine-tuning are the two mainstream options.

**Fine-tuning**: keep training the LLM on a task-specific dataset to lift its performance on that task.

The comparison table below is sourced from [[1](https://arxiv.org/abs/2312.10997)][[2](https://baoyu.io/translations/ai-paper/2312.10997-retrieval-augmented-generation-for-large-language-models-a-survey)]

| Aspect | RAG                                                          | Fine-tuning                                                  |
| -------- | ------------------------------------------------------------ | ------------------------------------------------------------ |
| Knowledge updates | Update the retrieval index directly, no retraining. Cheap to refresh, ideal for data that changes often. | Usually needs retraining to keep knowledge and data current. Expensive to refresh, suited to static data. |
| External knowledge | Excels at using external resources, especially documents or other structured/unstructured knowledge. | Restricted to knowledge baked into the training set; hard to plug in extra sources. |
| Data requirements | Needs a retrievable knowledge base; data quality still matters but annotation cost is lower. | Needs high-quality labelled task data; annotation cost is higher. |
| Cost | Low: mainly retrieval infrastructure plus index maintenance. | High: training compute plus storage, and repeat cost on every update. |
| Hallucination | Effectively reduced because answers are grounded in retrieved evidence. | Reduced within the trained task, but the model may still invent content outside it. |
| Interpretability | High: answers can be traced back to the retrieved sources. | Low: knowledge is implicit in the weights and hard to trace. |
| Flexibility | High: swap or extend the knowledge base at any time. | Low: a new task or new knowledge means retraining. |
| Typical use | Open-domain or enterprise Q&A, knowledge-intensive tasks, frequently updated content. | Style transfer, domain jargon, fixed task formats, low-latency edge deployment. |


## LangChain


### What LangChain is

ChatGPT's success sparked far more developer interest in building LLM applications on top of the OpenAI API or privately deployed models. Calling an LLM is simple, but building a complete application still takes a lot of custom work: API integration, interaction logic, data storage and so on.

To solve this, starting in 2022 many organisations and individuals released open-source projects aimed at **helping developers quickly build end-to-end LLM applications or workflows**. One of the most talked-about is the LangChain framework.

**LangChain is an open-source toolkit that leverages the power of LLMs to build all kinds of downstream applications. Its goal is to give LLM applications a common interface and thereby simplify application development.** Concretely, LangChain provides data awareness and environment interaction: it connects a language model to other data sources and lets it act on its environment.

With LangChain we can build the RAG application shown below. In the diagram, `each ellipse is one LangChain module`, such as the data collection or preprocessing module, and `each rectangle is one data state`, such as raw data or preprocessed data. Arrows show the direction of data flow from one module to the next. At every step LangChain offers a matching solution for the task at hand.

![Langchain overview](../../assets/langchain-01.png)

### The core components of LangChain

As an LLM development framework, LangChain ties together LLMs (chat models, embedding models, ...), vector databases, prompt layers, external knowledge and external agent tools, so you can compose LLM applications freely. It has six core components:

- **Model I/O**: the interface for interacting with language models
- **Data connection**: the interface for interacting with an application's data
- **Chains**: compose components into end-to-end applications. Later we build a `retrieval QA chain` to do retrieve-and-answer.
- **Memory**: persist application state across multiple chain runs;
- **Agents**: extend the model's reasoning ability, used for call sequences in complex applications;
- **Callbacks**: extend the model's reasoning ability, used for call sequences in complex applications;


# About this project

This project is a **hands-on introduction to Retrieval-Augmented Generation (RAG)** for beginners. LangChain is the core framework, and the notebook walks you through building a local knowledge-base Q&A system from scratch: RAG fundamentals, environment setup, document loading and splitting, vectorisation and vector storage, retrieval strategies, LLM generation, and a simple LLM-as-Judge evaluation.

## Project overview

The notebook follows the pipeline *data -> index -> retrieval -> generation -> evaluation*, so learners build an end-to-end view of a RAG system. It covers:

1. **Environment setup**: creating the uv virtual environment, installing dependencies, configuring the vLLM / Xinference model services;
2. **Knowledge-base construction**: load local PDF and Markdown documents with LangChain, split the text, embed the chunks, store them in a vector database;
3. **Retrieval and generation**: recall relevant chunks by vector search, then have the LLM generate a grounded answer;
4. **Evaluation loop**: use LLM-as-Judge to compare the RAG answer and the baseline answer on faithfulness, completeness and relevance;
5. **Where to go next**: a short intro to systematic evaluation frameworks such as Ragas as a reference for future optimisation.

After finishing this project you will be able to build a runnable, evaluable local knowledge-base Q&A system on your own, and you will understand what each step of the RAG pipeline does and where to tune it.


## Key libraries and environment notes for this project


> **Environment note**: this notebook runs in the `env_rag_uv` virtual environment created with `uv`, on Python 3.12, and the Jupyter kernel is also named `env_rag_uv`. The table below lists the core dependencies, what they are for, and the version actually installed. When running, the code cell below prints the real installed version of every library in your environment.


| Library | Version | Purpose |
|---|---|---|
| `langchain` | 1.4.7 | Main LangChain framework: chains, prompts, model interfaces and other core capabilities |
| `langchain-core` | 1.7.0 | LangChain core abstractions and runtime (Messages, Runnables, output parsers, ...) |
| `langchain-community` | 0.4.2 | Community integrations (document loaders such as `PyMuPDFLoader`) |
| `langchain-openai` | 1.7.2 | OpenAI-compatible interface; used here for the vLLM chat model and the Xinference embedding model |
| `langchain-milvus` | 0.4.0 | LangChain wrapper for the Milvus vector database (requires pymilvus 3.x) |
| `langchain-classic` | 1.0.8 | Legacy API compatibility (`RetrievalQA`, `ConversationalRetrievalChain`) |
| `langchain-text-splitters` | 1.1.3 | Text splitters (`RecursiveCharacterTextSplitter`, ...) |
| `langchain-cohere` | not installed | Cohere Rerank cloud reranking (optional, auto-skipped without a key) |
| `pymilvus` | 3.0.2 | Milvus Python SDK, used to connect to and operate the vector database |
| `PyMuPDF` | 2.0 | PDF parser, the underlying dependency of `PyMuPDFLoader` |
| `rank-bm25` | 0.2.2 | Hand-written BM25 keyword retrieval for teaching |
| `xinference-client` | 4.0.0 | Native Xinference REST client (optional; embeddings go through the OpenAI-compatible interface) |
| `python-dotenv` | 1.2.4 | Read service URLs and API keys from the `.env` file |
| `httpx` | 0.28.1 | HTTP client, used for vLLM / Xinference rerank / Jina rerank calls |
| `openai` | 3.37.0 | OpenAI SDK, required for OpenAI-compatible endpoints |
| `pandas` | 2.3.3 | Data handling and display |
| `numpy` | 2.5.4 | Numerical computing |
| `ipykernel` | 7.4.0 | Jupyter kernel needed to run the notebook |


In [ ]:
# show the versions actually installed in the env_rag_uv environment
from importlib.metadata import version, PackageNotFoundError

packages = [
    ("langchain", "LangChain main framework"),
    ("langchain-core", "LangChain core abstractions"),
    ("langchain-community", "community components and loaders"),
    ("langchain-openai", "OpenAI OpenAI-compatible API (vLLM / Xinference)"),
    ("langchain-milvus", "Milvus vector store wrapper"),
    ("langchain-classic", "legacy Chain API compatibility"),
    ("langchain-text-splitters", "text splitter"),
    ("langchain-cohere", "Cohere Rerank (optional)"),
    ("pymilvus", "Milvus Python SDK"),
    ("PyMuPDF", "PDF parsing"),
    ("rank-bm25", "BM25 keyword retrieval (teaching version)"),
    ("xinference-client", "Xinference native client (optional)"),
    ("python-dotenv", ".env read from the environment"),
    ("httpx", "HTTP client"),
    ("openai", "OpenAI SDK"),
    ("pandas", "data processing"),
    ("numpy", "numeric calculation"),
    ("ipykernel", "Jupyter Kernel"),
]

print(f"{'package':<35} {'version':<15} {'purpose'}")
print("-" * 80)
for name, desc in packages:
    try:
        ver = version(name)
    except PackageNotFoundError:
        ver = "not installed"
    print(f"{name:<35} {ver:<15} {desc}")


## Unified model configuration

This notebook uses one shared set of model and storage settings, all read from the repository root `.env`:

- **Embedding model**: Xinference `XINFERENCE_EMBEDDING_MODEL_ID` through `langchain_openai.OpenAIEmbeddings` (OpenAI-compatible `/v1/embeddings` endpoint).
- **Chat model**: vLLM `vLLM_QWEN_MODEL_ID` through `langchain.chat_models.init_chat_model` (OpenAI-compatible `/v1` endpoint).
- **Reranker model**: Xinference `XINFERENCE_RERANKER_MODEL_ID` through the OpenAI-compatible `/v1/rerank` endpoint.
- **Vector database**: `langchain_milvus.Milvus` connected to the Milvus Standalone at `MILVUS_URI`.

The full key reference for `.env` lives in `model-configuration.ipynb` at the repository root.

The cells below define `my_emb`, `my_llm`, the reranker constants and `MILVUS_URI` once. Later chapters sometimes redefine the same configuration inside their own cells so each section can run standalone.


In [ ]:
import os
import httpx
from dotenv import load_dotenv, find_dotenv
from langchain.chat_models import init_chat_model
from langchain_openai import OpenAIEmbeddings

# read the repository-root .env (walking upwards automatically when the notebook sits in a subdirectory)
_ = load_dotenv(find_dotenv())

# ── 1. LLM configuration (vLLM, OpenAI-compatible API)────────────────
my_llm = init_chat_model(
    model=os.environ["vLLM_QWEN_MODEL_ID"],
    model_provider="openai",
    base_url=os.environ["vLLM_QWEN_BASE_URL"],
    api_key=os.environ["vLLM_QWEN_API_KEY"],
    http_client=httpx.Client(trust_env=False),
    temperature=0.3,
    max_tokens=3000,
)

# ── 2. Embedding model configuration (Xinference, OpenAI-compatible API)───────
my_emb = OpenAIEmbeddings(
    base_url=os.environ["XINFERENCE_OPENAI_BASE_URL"],
    api_key=os.environ["XINFERENCE_API_KEY"],
    model=os.environ["XINFERENCE_EMBEDDING_MODEL_ID"],
)

# ── 3. Reranker model configuration (Xinference, /v1/rerank)────────────
RERANKER_BASE_URL = os.environ["XINFERENCE_OPENAI_BASE_URL"]
RERANKER_MODEL = os.environ["XINFERENCE_RERANKER_MODEL_ID"]

# ── 4. vector database address ────────────────────────────────────────
MILVUS_URI = os.environ["MILVUS_URI"]

print("LLM       :", os.environ["vLLM_QWEN_MODEL_ID"], "@", os.environ["vLLM_QWEN_BASE_URL"])
print("Embedding :", os.environ["XINFERENCE_EMBEDDING_MODEL_ID"], "@", os.environ["XINFERENCE_OPENAI_BASE_URL"])
print("Reranker  :", RERANKER_MODEL)
print("Milvus    :", MILVUS_URI)


# Data preprocessing


## Simple document preprocessing

To build our local knowledge base we must process local documents stored in various formats: read the documents, turn their content into embeddings with the embedding method described earlier, and build a vector database from them. The following sections start from concrete examples and show how to process local documents.
### Reading the data
If you have no knowledge-base files yet, put the corpus sources into the knowledge_db directory.


In [ ]:
# the corpus sources live in ../knowledge_path/VMAX-S/ next to this notebook; copy them straight out of the course bundle:
# !cp -r /path/to/DataSet/rag2026/VMAX-S/ ./knowledge_path/

**PDF documents as an example**
Use LangChain's PyMuPDFLoader to read the PDF files in the knowledge base. PyMuPDFLoader is the fastest PDF parser available: its result carries detailed metadata about the PDF and each page, and it returns one document per page.


In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader

# create a PyMuPDFLoader instance, passing the path of the PDF to load
loader = PyMuPDFLoader("../knowledge_path/VMAX-S/ZXVMAX-S\uff08V6.23\uff09\u4ea7\u54c1\u63cf\u8ff0\uff08\u4e0a\u7f51\u65e5\u5fd7\u4e1a\u52a1\uff09.pdf")  # the internet-access log manual

# call PyMuPDFLoader.load to read the PDF file
pdf_pages = loader.load()

The loaded document is stored in the `pdf_pages` variable:
- `pdf_pages` has type `List`
- printing `len(pdf_pages)` tells you how many pages the PDF contains


In [ ]:
print(f"type of the loaded variable: {type(pdf_pages)}",  f"this PDF contains {len(pdf_pages)} pages")

Each element of `pdf_pages` is one document of type `langchain_core.documents.base.Document`, which has two attributes:
- `page_content` holds the content of the document.
- `meta_data` holds descriptive metadata about the document.


In [ ]:
pdf_page = pdf_pages[20]
print(f"type of every element: {type(pdf_page)}.", 
    f"the descriptive metadata of this document:{pdf_page.metadata}", 
    f"inspect the content of this document:\n{pdf_page.page_content}", 
    sep="\n------\n")

In [ ]:
# assume the NLTK punkt resource is missing - punkt is the tokenizer NLTK needs. Fix it by downloading and installing punkt exactly as the error message suggests.
# import nltk
# nltk.download('punkt', quiet=True)

### Cleaning

The simple version only does generic whitespace normalisation: collapse runs of newlines and spaces into a single space. It is easy to understand, but it does not recognise headers and footers, headings, lists or the hard line breaks introduced by PDF layout. Treat it as the baseline for the optimised version.


In [ ]:
import re
from langchain_core.documents import Document

def simple_clean_document(doc: Document) -> Document:
    text = re.sub(r"\s+", " ", doc.page_content).strip()
    metadata = dict(doc.metadata)
    metadata["page_number"] = metadata.get("page", 0) + 1
    metadata["preprocess_version"] = "simple"
    return Document(page_content=text, metadata=metadata)

simple_cleaned_pages = [
    simple_clean_document(page) for page in pdf_pages if page.page_content.strip()
]
print(simple_cleaned_pages[20].page_content[:500])

### Splitting

The simple version splits purely by fixed character length, with no awareness of chapter boundaries.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

simple_text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=50,
)
simple_split_docs = simple_text_splitter.split_documents(simple_cleaned_pages)
for index, chunk in enumerate(simple_split_docs):
    chunk.metadata["chunk_index"] = index

print(f"chunks after the simple preprocessing split: {len(simple_split_docs)}")

## Optimised preprocessing

The previous section showed basic PDF loading and page structure. This section presents an optimised pipeline aimed at the ZXVMAX-S product manual: first restore the document structure and strip noise such as headers and footers, then build semantic boundaries from numbered chapter headings, then split only the chapters that are too long, while keeping source file, page number, business type, chapter heading and chunk index as metadata.


### Reading the data

The optimised version reads the same PDF, so the comparison below reflects only the difference in preprocessing strategy.


In [ ]:
optimized_loader = PyMuPDFLoader(
    "../knowledge_path/VMAX-S/ZXVMAX-S\uff08V6.23\uff09\u4ea7\u54c1\u63cf\u8ff0\uff08\u4e0a\u7f51\u65e5\u5fd7\u4e1a\u52a1\uff09.pdf",
)
optimized_pdf_pages = optimized_loader.load()
print(f"pages read by the optimised version: {len(optimized_pdf_pages)}")

### Cleaning


Cleaning a PDF is not just deleting newlines and spaces; the goal is to recover the document's original semantic structure. For the ZXVMAX-S product manual used here, the things that matter are headers and footers, standalone page numbers, the hard line breaks produced by PDF layout, and private-use-area bullet characters such as `.`.

This section works in three steps: normalise characters first, then filter noise by rule, then restore paragraph structure. Headings, lists, English terms and necessary spaces are preserved.


Define reusable cleaning rules


In [ ]:
import re
import unicodedata
from pathlib import Path
from langchain_core.documents import Document

# ZXVMAX-S the fixed footer of the document, for example:SJ-... | 2023-03-30 (R1.0)
FOOTER_PATTERN = re.compile(
    "^SJ-\d{14,}-\d+\s*\|\s*\d{4}-\d{2}-\d{2}\uff08R\d+\.\d+\uff09$"  # full-width brackets are literal corpus text
)
PAGE_NUMBER_PATTERN = re.compile(r"^\d{1,4}$")
HEADING_PATTERN = re.compile(
    r"^(?:\u7b2c[\u4e00\u4e8c\u4e09\u56db\u4e94\u516d\u4e03\u516b\u4e5d\u5341\u767e]+\u7ae0|\d+(?:\.\d+)*)(?:\s+|\u00a0).+"  # numbered chapter heading
)
BULLET_PATTERN = re.compile(r"^[\uf06c•▪▫▶●○]\s*")
REPEATED_HEADER_PATTERN = re.compile(
    r"^ZXVMAX-S(?: \u591a\u7ef4\u4ef7\u503c\u5206\u6790\u7cfb\u7edf)? \u4ea7\u54c1\u63cf\u8ff0\uff08.+\u4e1a\u52a1\uff09$"  # the manual cover title
)

In [ ]:
def normalize_line(line: str) -> str:
    """normalise Unicode, whitespace and bullet markers, without removing meaningful spaces."""
    # NFC only normalises Unicode composition; Chinese punctuation is never converted to ASCII punctuation
    line = unicodedata.normalize("NFC", line)
    line = line.replace("\u00a0", " ").replace("\u3000", " ")
    line = re.sub(r"[ \t]+", " ", line).strip()
    if BULLET_PATTERN.match(line):
        line = BULLET_PATTERN.sub("- " , line)
    return line


def should_join(previous: str, current: str) -> bool:
    """decide whether a line break came from PDF typesetting; headings, list items and complete sentences are never joined."""
    if not previous or not current:
        return False
    if current.startswith("- ") or previous.startswith("- "):
        return False
    if HEADING_PATTERN.match(previous) or HEADING_PATTERN.match(current):
        return False
    # unnumbered sub-headings in the manuals are usually short, such as fault tolerance or collection layer”
    if len(previous) <= 20 and not previous.endswith(("\uff0c", "\u3001", "\uff1a", "\uff1b")):
        return False
    if previous.endswith(("\u3002", "\uff01", "\uff1f", "\uff1b", "\uff1a", "\uff09", "\u3011")):
        return False
    return True


def clean_pdf_document(doc: Document) -> Document:
    """clean one PDF page while keeping the source and page metadata intact."""
    raw_text = doc.page_content.replace("\r\n", "\n").replace("\r", "\n")
    lines = []
    pending_bullet = False

    for raw_line in raw_text.split("\n"):
        line = normalize_line(raw_line)
        if not line:
            continue
        # PDF bullet markers often sit alone on a line in PDFs, so merge one with the list heading on the next line
        if line == "- ":
            pending_bullet = True
            continue
        if pending_bullet:
            line = "- " + line
            pending_bullet = False
        if FOOTER_PATTERN.match(line) or PAGE_NUMBER_PATTERN.match(line):
            continue
        if REPEATED_HEADER_PATTERN.match(line):
            continue
        lines.append(line)

    paragraphs = []
    for line in lines:
        if paragraphs and should_join(paragraphs[-1], line):
            paragraphs[-1] += line
        else:
            paragraphs.append(line)

    metadata = dict(doc.metadata)
    metadata["source_name"] = Path(metadata.get("source", "")).name
    metadata["page_number"] = metadata.get("page", 0) + 1
    metadata["cleaned"] = True

    return Document(
        page_content="\n".join(paragraphs).strip(),
        metadata=metadata,
    )


# clean every page of the optimised version; keep the raw pages so before/after can be compared
cleaned_pdf_pages = [
    clean_pdf_document(page)
    for page in optimized_pdf_pages
    if page.page_content.strip()
]
cleaned_pdf_page = cleaned_pdf_pages[20]

# before/after cleaning comparison and basic quality checks
print("===== before cleaning (first 500 characters)=====")
print(optimized_pdf_pages[20].page_content[:500])
print("\n===== after cleaning (first 500 characters)=====")
print(cleaned_pdf_page.page_content[:500])
print("\n===== quality statistics =====")
print("raw page count: ", len(optimized_pdf_pages))
print("valid pages after cleaning: ", len(cleaned_pdf_pages))
print("raw character count: ", sum(len(page.page_content) for page in optimized_pdf_pages))
print("characters after cleaning: ", sum(len(page.page_content) for page in cleaned_pdf_pages))
print("pages still holding private bullet markers: ", sum("\uf06c" in page.page_content for page in cleaned_pdf_pages))
print("metadata of the current page: ", cleaned_pdf_page.metadata)

### Splitting

A single document is usually longer than the context a model supports, so retrieved knowledge would exceed the model's processing capacity. When building a vector knowledge base we therefore split documents into chunks, by length or by a fixed rule, embed each chunk, and store the vectors in a vector database.

At retrieval time the chunk is the atomic unit: each query returns k chunks as the knowledge the model may use to answer the user's question, and k is freely configurable.

Every LangChain text splitter splits according to `chunk_size` (chunk length) and `chunk_overlap` (overlap between neighbouring chunks).

![image.png](../../assets/rag-08.png)

* chunk_size is the number of characters or tokens (words, sentences, ...) in each chunk

* chunk_overlap is the number of characters two chunks share, used to keep context coherent so splitting does not lose information


**LangChain offers several splitters; they differ in how chunk boundaries are decided, what characters/tokens a chunk consists of, and how chunk size is measured**

- RecursiveCharacterTextSplitter(): splits text by characters, recursively trying different separators.
- CharacterTextSplitter(): splits text by characters.
- MarkdownHeaderTextSplitter(): splits a Markdown file based on the specified headings.
- TokenTextSplitter(): splits text by tokens.
- SentenceTransformersTokenTextSplitter(): splits text by tokens.
- Language(): for CPP, Python, Ruby, Markdown and similar.
- NLTKTextSplitter(): splits text into sentences using NLTK (Natural Language Toolkit).
- SpacyTextSplitter(): splits text into sentences using SpaCy.


In [ ]:
''' 
* RecursiveCharacterTextSplitter recursive character text splitting
RecursiveCharacterTextSplitter splits recursively on different characters (in this priority order: ["\n\n", "\n", " ", ""]),
    this keeps semantically related content together for as long as possible
RecursiveCharacterTextSplitterthe four parameters that matter:

* separators - array of separator strings
* chunk_size - character limit per document
* chunk_overlap - length of the overlap between two chunks
* length_function - length function
'''
#import the text splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
# length of a single chunk in the knowledge base
CHUNK_SIZE = 512

# overlap between neighbouring chunks in the knowledge base
OVERLAP_SIZE = 50

In [ ]:
# the second stage only splits oversized sections; the section boundaries of stage one are detected by the function below
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=OVERLAP_SIZE,
    separators=["\n\n", "\n", "\u3002", "\uff1b", "\uff01", "\uff1f", "\uff0c", ""],  # Chinese sentence punctuation
)

SECTION_TITLE_PATTERN = re.compile(
    r"^(?:\u7b2c[\u4e00\u4e8c\u4e09\u56db\u4e94\u516d\u4e03\u516b\u4e5d\u5341\u767e]+\u7ae0(?:\s+.+)?|\d+(?:\.\d+)+(?:\s+.+)|\d+\s+[^0-9].+)"
)
BUSINESS_TYPE_PATTERN = re.compile("\uff08([^\uff08\uff09]+service)\uff09")


def is_section_title(line: str) -> bool:
    # table-of-contents entries contain runs of dot leaders and must not count as section boundaries
    return not re.search("[.\uff0e\u00b7\u2026]{3,}", line) and bool(SECTION_TITLE_PATTERN.match(line))


def extract_business_type(source_name: str) -> str:
    match = BUSINESS_TYPE_PATTERN.search(source_name)
    return match.group(1) if match else "unknown service"


def build_section_documents(pages):
    """detect sections per document and page; across pages, inherit the last section seen on the previous page."""
    section_docs = []
    current_section_by_source = {}

    for page in pages:
        metadata = dict(page.metadata)
        source_name = metadata.get("source_name") or Path(metadata.get("source", "")).name
        current_section = current_section_by_source.get(source_name, "unidentified section")
        buffer = []

        def flush_buffer():
            if not buffer:
                return
            block_text = "\n".join(buffer).strip()
            # a page tail can hold only the next section heading while the body starts on the next page; this avoids empty heading-only chunks
            if block_text == current_section:
                buffer.clear()
                return
            block_metadata = dict(metadata)
            block_metadata.update({
                "source_name": source_name,
                "business_type": extract_business_type(source_name),
                "section_title": current_section,
            })
            section_docs.append(Document(
                page_content=block_text,
                metadata=block_metadata,
            ))
            buffer.clear()

        for line in page.page_content.splitlines():
            if is_section_title(line):
                flush_buffer()
                current_section = line
                current_section_by_source[source_name] = current_section
            buffer.append(line)
        flush_buffer()

    return section_docs


def split_documents_by_section(pages):
    """section-first, length-bounded splitting that gives every source a stable chunk_index."""
    from collections import defaultdict

    section_docs = build_section_documents(pages)
    chunks = text_splitter.split_documents(section_docs)
    source_counters = defaultdict(int)

    for chunk in chunks:
        source_name = chunk.metadata["source_name"]
        chunk.metadata["chunk_index"] = source_counters[source_name]
        source_counters[source_name] += 1

        # titles are fed into the embedding/BM25 text as well, which lifts recall for section-name questions
        section_title = chunk.metadata["section_title"]
        if section_title != "unidentified section" and not chunk.page_content.startswith(section_title):
            chunk.page_content = f"{section_title}\n{chunk.page_content}"

    return chunks

In [ ]:
# split on section headings first, then split any section longer than CHUNK_SIZE a second time
split_docs = split_documents_by_section(cleaned_pdf_pages)
print(f"chunks after the section-aware split: {len(split_docs)}")
print("metadata example: ", {
    key: split_docs[0].metadata.get(key)
    for key in ["source_name", "page_number", "business_type", "section_title", "chunk_index"]
})

In [ ]:
split_docs[50].page_content

In [ ]:
required_metadata = {"source_name", "page_number", "business_type", "section_title", "chunk_index"}
print(f"characters after splitting: {sum(len(doc.page_content) for doc in split_docs)}")
print("metadata complete: ", all(
    required_metadata.issubset(doc.metadata) for doc in split_docs
))
print("dot-leader lines mistaken for headings: ", sum(
    bool(re.search("[.\uff0e\u00b7\u2026]{3,}", doc.metadata["section_title"]))
    for doc in split_docs
))

Note: document splitting usually sets the floor of retrieval quality. This section uses a two-stage strategy: first recover semantic boundaries from numbered chapter headings, then apply a second split to chapters that are too long using `chunk_size` and `chunk_overlap`. Dot leaders in the table of contents are excluded, and body text that spans pages inherits the chapter heading from the previous page. Every chunk also stores source file, page number, business type, chapter heading and in-document index, which makes filtered retrieval, citation tracing and issue localisation possible.


### Simple vs optimised preprocessing

The cell below compares the same fixed block: page 21 of the same document, the block containing "7.3 data reliability". The simple version slices by length, the optimised version organises by chapter and carries richer retrieval and provenance metadata.


In [ ]:
COMPARE_PAGE_NUMBER = 21
COMPARE_SECTION = "7.3 data reliability"

simple_example = next(
    doc for doc in simple_split_docs
    if doc.metadata.get("page_number") == COMPARE_PAGE_NUMBER
    and "data reliability" in doc.page_content
)
optimized_example = next(
    doc for doc in split_docs
    if doc.metadata.get("page_number") == COMPARE_PAGE_NUMBER
    and doc.metadata.get("section_title") == COMPARE_SECTION
)

print("===== simple preprocessing chunks =====")
print(simple_example.page_content)
print("metadata:", simple_example.metadata)
print("\n===== optimised preprocessing chunks =====")
print(optimized_example.page_content)
print("metadata:", {
    key: optimized_example.metadata.get(key)
    for key in ["source_name", "page_number", "business_type", "section_title", "chunk_index"]
})
print("\n===== comparison conclusion =====")
print("characters in the simple version: ", len(simple_example.page_content))
print("characters in the optimised version: ", len(optimized_example.page_content))
print("section titles of the optimised version: ", optimized_example.metadata["section_title"])

## Batch preprocessing


Process every file in the folder in one pass


In [ ]:
import os

# collect every file path under folder_path into file_paths
file_paths = []
folder_path = '../knowledge_path/VMAX-S'
for root, dirs, files in os.walk(folder_path):
    for file in files:
        file_path = os.path.join(root, file)
        file_paths.append(file_path)
print(file_paths)

from langchain_community.document_loaders import PyMuPDFLoader
# from langchain_community.document_loaders import UnstructuredMarkdownLoader

# walk the file paths and keep the instantiated loaders in loaders
loaders = []

for file_path in file_paths:
    file_type = file_path.split('.')[-1]
    if file_type == 'pdf':
        loaders.append(PyMuPDFLoader(file_path))
    else:
        print(f"Unsupported file type: {file_type} for file {file_path}")


# download the file and store it intext
# load all document content into texts
texts = []
for loader in loaders:
    texts.extend(loader.load())  # key step: initialise texts


# reuse the cleaning function defined earlier so single-document and batch rules cannot drift apart
raw_texts = texts
texts = [clean_pdf_document(doc) for doc in raw_texts if doc.page_content.strip()]
print(f"raw page count: {len(raw_texts)}, valid pages after cleaning: {len(texts)}")

In [ ]:
''' 
* RecursiveCharacterTextSplitter recursive character text splitting
RecursiveCharacterTextSplitter splits recursively on different characters (in this priority order: ["\n\n", "\n", " ", ""]),
    this keeps semantically related content together for as long as possible
RecursiveCharacterTextSplitterthe four parameters that matter:

* separators - array of separator strings
* chunk_size - character limit per document
* chunk_overlap - length of the overlap between two chunks
* length_function - length function
'''
#import the text splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
# length of a single chunk in the knowledge base
CHUNK_SIZE = 512

# overlap between neighbouring chunks in the knowledge base
OVERLAP_SIZE = 50

In [ ]:
# use the recursive character text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=OVERLAP_SIZE,
    separators=["\n\n", "\n", "\u3002", "\uff1b", "\uff01", "\uff1f", "\uff0c", ""],  # Chinese sentence punctuation
)
# text_splitter.split_text(text.page_content[0:1000])

In [ ]:
split_docs = split_documents_by_section(texts)
print(f"chunks after splitting: {len(split_docs)}")

In [ ]:
print(f"characters after splitting (a rough proxy for tokens): {sum([len(doc.page_content) for doc in split_docs])}")

In [ ]:
split_docs[300].page_content

# Building the vector database


### Installing Milvus


Milvus Standalone is the single-server deployment. All Milvus Standalone components are packaged into one [Docker image](https://milvus.io/docs/install_standalone-docker.md), which makes deployment very convenient. If you have production workloads but do not want Kubernetes, running Milvus Standalone on a single server with enough memory is a good choice. Milvus Standalone also supports high availability through primary-replica replication.

**Running Milvus with Docker Compose (Linux)**

The Milvus repository ships a Docker Compose configuration file. To install Milvus with Docker Compose, run

```shell
# Download the configuration file
$ wget https://github.com/milvus-io/milvus/releases/download/v3.0.3/milvus-standalone-docker-compose.yaml -O docker-compose.yaml


# Start Milvus
$ sudo docker compose -f docker-compose.yaml up -d

Creating milvus-etcd  ... done
Creating milvus-minio ... done
Creating milvus-standalone ... done
```

- If the command above fails, check whether the system has Docker Compose V1 installed. If so, migrate to Docker Compose V2 following [this page](https://docs.docker.com/compose/).
- If you hit problems pulling the images, contact us at [community@zilliz.com](mailto:community@zilliz.com) with the details and we will help.

After Milvus starts,

- the containers milvus-standalone, milvus-minio and milvus-etcd are running.
  - The **milvus-etcd** container exposes no ports to the host and maps its data to **volumes/etcd** in the current folder.
  - The **milvus-minio** container serves local ports **9090** and **9091** with the default credentials and maps its data to **volumes/minio** in the current folder.
  - The **milvus-standalone** container serves local port **19530** with default settings and maps its data to **volumes/milvus** in the current folder.

Check that the containers are up with:

```shell
$ sudo docker-compose ps

      Name                     Command                  State                            Ports
--------------------------------------------------------------------------------------------------------------------
milvus-etcd         etcd -advertise-client-url ...   Up             2379/tcp, 2380/tcp
milvus-minio        /usr/bin/docker-entrypoint ...   Up (healthy)   9000/tcp
milvus-standalone   /tini -- milvus run standalone   Up             0.0.0.0:19530->19530/tcp, 0.0.0.0:9091->9091/tcp
```

You can also open the Milvus WebUI at `http://127.0.0.1:9091/webui/` to learn more about the Milvus instance. See [Milvus WebUI](https://milvus.io/docs/milvus-webui.md) for details.

**Stopping and removing Milvus**

Stop and remove the containers as follows

```shell
# Stop Milvus
$ sudo docker compose down

# Delete service data
$ sudo rm -rf volumes
```


**Attu (the Milvus GUI)**

https://github.com/zilliztech/attu

```shell
# Attu v3 takes the Milvus address through MILVUS_ADDRESS; use host.docker.internal to reach the host from inside the container
docker run -d --name attu-container   -p 9092:3000   -e MILVUS_ADDRESS=http://host.docker.internal:19530   -v attu-data:/data   zilliz/attu:v3.0.0
```

Then open `http://localhost:9092` and enter the same Milvus address on the login page; no username or password is required.

**Milvus user guide**

- [Databases](https://milvus.io/docs/manage_databases.md)
- Collections
- Schema and data fields
- Insert and delete
- Indexes
- Search and Rerankers

> Reference: https://milvus.io/docs/manage_databases.md


## Preprocessing from the previous chapter


In [ ]:
#!/usr/bin/env python
# coding: utf-8

# process every file in the folder in batch
import os

# collect every file path under folder_path into file_paths
file_paths = []
folder_path = '../knowledge_path/VMAX-S'
for root, dirs, files in os.walk(folder_path):
    for file in files:
        file_path = os.path.join(root, file)
        file_paths.append(file_path)
print(file_paths)

from langchain_community.document_loaders import PyMuPDFLoader
# from langchain_community.document_loaders import UnstructuredMarkdownLoader

# walk the file paths and keep the instantiated loaders in loaders
loaders = []

for file_path in file_paths:
    file_type = file_path.split('.')[-1]
    if file_type == 'pdf':
        loaders.append(PyMuPDFLoader(file_path))
    else:
        print(f"Unsupported file type: {file_type} for file {file_path}")

# download the file and store it intext
# load all document content into texts
texts = []
for loader in loaders:
    texts.extend(loader.load())  # key step: initialise texts


# reuse the cleaning function defined earlier
raw_texts = texts
texts = [clean_pdf_document(doc) for doc in raw_texts if doc.page_content.strip()]
print(f"raw page count: {len(raw_texts)}, valid pages after cleaning: {len(texts)}")




#import the text splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
''' 
* RecursiveCharacterTextSplitter recursive character text splitting
RecursiveCharacterTextSplitter splits recursively on different characters (in this priority order: ["\n\n", "\n", " ", ""]),
    this keeps semantically related content together for as long as possible
RecursiveCharacterTextSplitterthe four parameters that matter:

* separators - array of separator strings
* chunk_size - character limit per document
* chunk_overlap - length of the overlap between two chunks
* length_function - length function
'''


# length of a single chunk in the knowledge base
CHUNK_SIZE = 512

# overlap between neighbouring chunks in the knowledge base
OVERLAP_SIZE = 50


# use the recursive character text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=OVERLAP_SIZE,
    separators=["\n\n", "\n", "\u3002", "\uff1b", "\uff01", "\uff1f", "\uff0c", ""],  # Chinese sentence punctuation
)


split_docs = split_documents_by_section(texts)
print(f"chunks after splitting: {len(split_docs)}")



print(f"characters after splitting (a rough proxy for tokens): {sum([len(doc.page_content) for doc in split_docs])}")



split_docs[300].page_content



## Preparing the embedding model


In [ ]:
import sys
print(sys.executable)

import langchain_core
print(langchain_core.__version__)

import langchain_core.messages as m
print(m.__file__)

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())

from langchain_openai import OpenAIEmbeddings

# initialise the embedding model
my_emb = OpenAIEmbeddings(
    base_url=os.environ["XINFERENCE_OPENAI_BASE_URL"],
    api_key=os.environ["XINFERENCE_API_KEY"],
    model=os.environ["XINFERENCE_EMBEDDING_MODEL_ID"],
)


In [ ]:
# alternative: the native Xinference REST client (requires xinference-client)
# from xinference_client import RESTfulClient
#
# xf = RESTfulClient(
#     os.environ["XINFERENCE_BASE_URL"],
#     api_key=os.environ["XINFERENCE_API_KEY"],
# )
# for model_uid, info in xf.list_models().items():
#     print(model_uid, info["model_type"], info.get("dimensions"))


## Building the Milvus vector store - simple retrieval version



In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())

from pymilvus import MilvusClient
from langchain_milvus import Milvus

# Milvus connection arguments. The Quickstart uses MilvusClient(uri=...) connect to the database; here it connects to a remote Milvus Standalone.
connection_args = {
    "uri": os.environ["MILVUS_URI"],
}

_milvus_client = MilvusClient(**connection_args)

# the simple collection holds a single dense vector field, i.e. the single-vector semantic search from the Milvus Quickstart.
simple_collection_name = "ZXVMAXS"
dense_index_param = {
    "metric_type": "COSINE",
    "index_type": "HNSW",
}

batch_size = 30

try:
    total_batches = (len(split_docs) + batch_size - 1) // batch_size
    vectordb = None

    for batch_num in range(total_batches):
        start_idx = batch_num * batch_size
        end_idx = min((batch_num + 1) * batch_size, len(split_docs))
        batch_docs = split_docs[start_idx:end_idx]

        print(f"processing  {batch_num + 1}/{total_batches}  documents (document {start_idx}-{end_idx - 1})")

        if batch_num == 0:
            vectordb = Milvus.from_documents(
                documents=batch_docs,
                embedding=my_emb,
                collection_name=simple_collection_name,
                connection_args=connection_args,
                vector_field="dense",
                text_field="text",
                index_params=dense_index_param,
                drop_old=True,
            )
        else:
            vectordb.add_documents(batch_docs)

        print(f"batch {batch_num + 1}: documents were imported into the Milvus simple collection")

    # the search only needs text and metadata fields, not the raw dense vectors.
    vectordb.fields = [field for field in vectordb.fields if field not in {"dense", "sparse", "vector"}]
    print(f"all documents imported into the simple collection: {simple_collection_name}")
    print(f"vector fields: {vectordb.vector_fields}")
    print(f"output fields: {vectordb.fields}")

except Exception as e:
    print(f"an error occurred while building the Milvus simple collection: {e}")


## Building the Milvus vector store - hybrid retrieval version


Follow the basic flow of the [Milvus Quickstart](https://milvus.io/docs/v2.6.x/quickstart.md): connect to Milvus, create a collection, write vector and scalar fields, and at search time return only the text and metadata that need to be displayed. With the LangChain wrapper this maps to:

- `dense`: produced by the embedding model, used for semantic similarity search;
- `sparse`: produced server-side by the Milvus BM25 built-in function from the raw text, used for keyword search;
- at retrieval time the dense and sparse result sets are fused with `ranker_type="weighted"` or `ranker_type="rrf"`;
- `sparse` is the sparse vector field produced by the BM25 function and cannot be returned as raw data, so `dense/sparse/vector` must be excluded from `fields` before searching.

`BM25BuiltInFunction` requires no corpus statistics on the client: text analysis, sparse vector generation and BM25 scoring all happen inside Milvus. Note that Milvus Lite does not support this built-in full-text search capability; you need Milvus Standalone, Milvus Distributed, or a compatible service.



In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())

from pymilvus import MilvusClient
from langchain_milvus import Milvus, BM25BuiltInFunction

connection_args = {
    "uri": os.environ["MILVUS_URI"],
}

_milvus_client = MilvusClient(**connection_args)

# the hybrid collection uses its own name so its schema cannot clash with the single-vector simple collection.
hybrid_collection_name = "ZXVMAXS_HYBRID"

dense_index_param = {
    "metric_type": "COSINE",
    "index_type": "HNSW",
}
sparse_index_param = {
    "metric_type": "BM25",
    "index_type": "AUTOINDEX",
}

batch_size = 30

try:
    total_batches = (len(split_docs) + batch_size - 1) // batch_size
    vectordb_hybrid = None

    for batch_num in range(total_batches):
        start_idx = batch_num * batch_size
        end_idx = min((batch_num + 1) * batch_size, len(split_docs))
        batch_docs = split_docs[start_idx:end_idx]

        print(f"processing  {batch_num + 1}/{total_batches}  documents (document {start_idx}-{end_idx - 1})")

        if batch_num == 0:
            vectordb_hybrid = Milvus.from_documents(
                documents=batch_docs,
                embedding=my_emb,
                builtin_function=BM25BuiltInFunction(output_field_names="sparse"),
                vector_field=["dense", "sparse"],
                collection_name=hybrid_collection_name,
                connection_args=connection_args,
                consistency_level="Bounded",
                index_params=[dense_index_param, sparse_index_param],
                drop_old=True,
            )
        else:
            vectordb_hybrid.add_documents(batch_docs)

        print(f"batch {batch_num + 1}: documents were imported into the Milvus hybrid collection")

    # the search only returns text and metadata fields; Milvus will not return the sparse raw data produced by the BM25 function.
    vectordb_hybrid.fields = [
        field for field in vectordb_hybrid.fields
        if field not in {"dense", "sparse", "vector"}
    ]
    print(f"all documents imported into the hybrid collection: {hybrid_collection_name}")
    print(f"vector fields: {vectordb_hybrid.vector_fields}")
    print(f"output fields: {vectordb_hybrid.fields}")

except Exception as e:
    print(f"an error occurred while building the Milvus hybrid collection: {e}")


The next cell runs one quick retrieval against the same Milvus collection. The order of `weights` matches `vector_field=["dense", "sparse"]`: the first weight controls semantic vector search, the second controls BM25 keyword search.


In [ ]:
query = "ZXVMAX-SWhich features does the internet-access log have?"

# Milvus sparse/dense vector fields cannot be returned as raw data; keep only the ordinary fields before hybrid search.
vectordb_hybrid.fields = [
    field for field in vectordb_hybrid.fields
    if field not in {"dense", "sparse", "vector"}
]

hybrid_docs = vectordb_hybrid.similarity_search(
    query,
    k=5,
    ranker_type="weighted",
    ranker_params={"weights": [0.6, 0.4]},
)

for i, doc in enumerate(hybrid_docs, 1):
    source = doc.metadata.get("source") or doc.metadata.get("file_path") or "unknown source"
    page = doc.metadata.get("page", "-")
    print(f"===== hybrid retrieval result {i} | source={source} | page={page} =====")
    print(doc.page_content[:500])
    print()


In [ ]:
# you can also fuse the two recalls with RRF (Reciprocal Rank Fusion), which is handy when you do not want to tune weights by hand.
# Milvus sparse/dense vector fields cannot be returned as raw data; keep only the ordinary fields before hybrid search.
vectordb_hybrid.fields = [
    field for field in vectordb_hybrid.fields
    if field not in {"dense", "sparse", "vector"}
]

rrf_docs = vectordb_hybrid.similarity_search(
    query,
    k=5,
    ranker_type="rrf",
    ranker_params={"k": 100},
)

for i, doc in enumerate(rrf_docs, 1):
    source = doc.metadata.get("source") or doc.metadata.get("file_path") or "unknown source"
    page = doc.metadata.get("page", "-")
    print(f"===== RRFhybrid retrieval result {i} | source={source} | page={page} =====")
    print(doc.page_content[:500])
    print()


## Connecting to the database for a test



In [ ]:
from dotenv import load_dotenv, find_dotenv
from pymilvus import MilvusClient

_ = load_dotenv(find_dotenv())

def list_all_collections(uri: str):
    """list every collection."""
    try:
        client = MilvusClient(uri=uri)
        collections = client.list_collections()
        print("all collections:")
        for i, coll_name in enumerate(collections, 1):
            print(f"{i}. {coll_name}")
        return collections
    except Exception as e:
        print("failed to list collections:", e)
        return []

# execute
collections = list_all_collections(os.environ["MILVUS_URI"])
print(f"found  {len(collections)} collections")


## Comparing the simple and hybrid retrieval versions

Following the retrieval flow of the [Milvus Quickstart](https://milvus.io/docs/v2.6.x/quickstart.md): write document vectors and metadata into a collection, then issue a `search` with the same query and inspect the Top-K hits with their text and metadata. We apply the same idea to compare the two Milvus collections created above:

- **simple retrieval version `ZXVMAXS`**: uses only the `dense` vector field, so recall relies mainly on semantic similarity;
- **hybrid retrieval version `ZXVMAXS_HYBRID`**: uses the `dense` semantic vector plus the `sparse` BM25 keyword vector, then fuses the two result sets with a weighted ranker.

When the query contains keywords such as product names, alarm codes or feature names, hybrid retrieval usually pushes the exact keyword hits to the front. When the query is phrased in natural-language semantics, plain vector search and hybrid search often overlap heavily.



In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())

from pymilvus import MilvusClient
from langchain_milvus import Milvus, BM25BuiltInFunction
from langchain_openai import OpenAIEmbeddings

my_emb = OpenAIEmbeddings(
    base_url=os.environ["XINFERENCE_OPENAI_BASE_URL"],
    api_key=os.environ["XINFERENCE_API_KEY"],
    model=os.environ["XINFERENCE_EMBEDDING_MODEL_ID"],
)

connection_args = {
    "uri": os.environ["MILVUS_URI"],
}

_milvus_client = MilvusClient(**connection_args)

vectordb = Milvus(
    embedding_function=my_emb,
    collection_name="ZXVMAXS",  # simple collection: a single dense vector field
    connection_args=connection_args,
    vector_field="dense",
    text_field="text",
)

vectordb_hybrid = Milvus(
    embedding_function=my_emb,
    builtin_function=BM25BuiltInFunction(output_field_names="sparse"),
    collection_name="ZXVMAXS_HYBRID",  # hybrid collection: dense + sparse two vector fields
    connection_args=connection_args,
    vector_field=["dense", "sparse"],
    text_field="text",
    consistency_level="Bounded",
)

# Milvus vector fields must not be returned as raw data, above all the sparse field generated by BM25.
# dropping dense/sparse/vector explicitly is sturdier than relying on vector_fields alone.
for vectorstore in [vectordb, vectordb_hybrid]:
    vectorstore.fields = [
        field for field in vectorstore.fields
        if field not in {"dense", "sparse", "vector"}
    ]


In [ ]:
comparison_query = "ZXVMAX-SWhich features does the internet-access log have?"
k = 5

# Milvus sparse/dense vector fields cannot be returned as raw data; keep only the ordinary fields before searching.
for vectorstore in [vectordb, vectordb_hybrid]:
    vectorstore.fields = [
        field for field in vectorstore.fields
        if field not in {"dense", "sparse", "vector"}
    ]

print("simple retrieval output fields: ", vectordb.fields)
print("hybrid retrieval output fields: ", vectordb_hybrid.fields)

simple_docs = vectordb.similarity_search(comparison_query, k=k)
hybrid_docs = vectordb_hybrid.similarity_search(
    comparison_query,
    k=k,
    ranker_type="weighted",
    ranker_params={"weights": [0.6, 0.4]},  # dense weight 0.6, BM25 sparse weight 0.4
)

def doc_key(doc):
    source = str(doc.metadata.get("source") or doc.metadata.get("file_path") or "unknown source").strip()
    page = str(doc.metadata.get("page", "-")).strip()
    return source, page

def print_results(title, docs):
    print(f"\n===== {title} =====")
    displayed = set()
    row = 1
    for doc in docs:
        source = str(doc.metadata.get("source") or doc.metadata.get("file_path") or "unknown source").strip()
        page = str(doc.metadata.get("page", "-")).strip()
        display_key = (source, page)
        if display_key in displayed:
            continue
        displayed.add(display_key)
        preview = doc.page_content[:180].replace("\n", " ")
        print(f"{row}. source={source} | page={page}")
        print(f"   {preview}")
        row += 1

print(f"Query: {comparison_query}")
print_results("simple retrieval Top-K (dense semantic search)", simple_docs)
print_results("hybrid retrieval Top-K (dense + BM25 sparse weighted fusion)", hybrid_docs)

simple_keys = {doc_key(doc): i + 1 for i, doc in enumerate(simple_docs)}
hybrid_keys = {doc_key(doc): i + 1 for i, doc in enumerate(hybrid_docs)}
common_keys = set(simple_keys) & set(hybrid_keys)
only_simple = set(simple_keys) - set(hybrid_keys)
only_hybrid = set(hybrid_keys) - set(simple_keys)

print("\n===== Top-K difference statistics =====")
print(f"documents found by both: {len(common_keys)}/{k}")
print(f"only simple retrieval hit: {len(only_simple)}")
print(f"only hybrid retrieval hit: {len(only_hybrid)}")

if only_hybrid:
    print("\nan example only hybrid retrieval found:")
    hybrid_doc_by_key = {doc_key(doc): doc for doc in hybrid_docs}
    for key in list(only_hybrid)[:3]:
        doc = hybrid_doc_by_key[key]
        source, page = key
        preview = doc.page_content[:120].replace("\n", " ")
        print(f"- source={source} | page={page} | {preview}")

print("\ncomparison conclusion: ")
print("- simple retrieval only looks at dense semantic similarity, which suits heavily paraphrased questions.")
print("- hybrid retrieval fuses dense semantic recall with BM25 keyword recall, which suits questions naming products, modules, alarm codes or proper nouns.")
print("- a low overlap means the keyword channel visibly changed the ranking or the candidate set; a high overlap means semantic recall is already stable for this query.")


### Reading the comparison

In this real-environment run, `ZXVMAXS` and `ZXVMAXS_HYBRID` were both built from the same `split_docs`, so a high Top-K overlap between them is expected. The query "What features does the ZXVMAX-S internet-access log have?" contains both the product name and feature keywords, and `dense` semantic search already recalls the internet-access-log pages reliably, so the two versions share 4 hits.

Hybrid retrieval additionally hits the feature-overview page of the "end-to-end services" document, which the simple version misses. That shows the BM25 keyword channel reinforces literal matches on `ZXVMAX-S`, `feature` and similar terms before fusing with the dense recall. The result illustrates the difference:

- simple retrieval leans on semantic similarity and suits questions phrased as natural language;
- hybrid retrieval considers both semantic similarity and keyword matches, so it suits questions containing proper nouns such as product names, module names, alarm codes and interface names;
- retrieval returns only ordinary fields such as `text/source/page` and never the `dense/sparse` vector fields. This matches the Milvus Quickstart practice of controlling returned fields through `output_fields`, and it avoids the `not allowed to retrieve raw data of field sparse` error.



## Building the retrieval QA chain - baseline


This chapter builds the RAG QA chain on the **simple retrieval vector store `ZXVMAXS`** created in the previous chapter. The flow splits into four steps:

1. connect to Milvus and load the simple retrieval collection that holds only the `dense` vector field;
2. embed the user question and recall the Top-K relevant chunks from Milvus;
3. insert the recalled chunks into the prompt and constrain the LLM to answer from context only;
4. compare the answer "with knowledge base" against the answer "without knowledge base" to see how RAG affects specialist questions and hallucination.

This chapter deliberately does not use BM25 sparse hybrid retrieval, so that the most basic dense-vector retrieval QA chain can be taught clearly on its own.



## Loading the Milvus vector store

Here we load the simple retrieval collection `ZXVMAXS` created in the previous chapter. Three things matter:

- `embedding_function` must be the same embedding model used when the vector store was built;
- the collection's vector field is named `dense` and its text field is named `text`, so the LangChain default `vector` cannot be used;
- retrieval results only need the text and metadata fields, never the raw vector field, so `dense/sparse/vector` is excluded from `fields` below.



In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())

from pymilvus import MilvusClient
from langchain_milvus import Milvus
from langchain_openai import OpenAIEmbeddings

my_emb = OpenAIEmbeddings(
    base_url=os.environ["XINFERENCE_OPENAI_BASE_URL"],
    api_key=os.environ["XINFERENCE_API_KEY"],
    model=os.environ["XINFERENCE_EMBEDDING_MODEL_ID"],
)

connection_args = {
    "uri": os.environ["MILVUS_URI"],
}

_milvus_client = MilvusClient(**connection_args)

vectordb = Milvus(
    embedding_function=my_emb,
    collection_name="ZXVMAXS",
    connection_args=connection_args,
    vector_field="dense",
    text_field="text",
)

# the search only needs text and metadata fields, not the raw dense vectors.
vectordb.fields = [
    field for field in vectordb.fields
    if field not in {"dense", "sparse", "vector"}
]

print("vector fields: ", vectordb.vector_fields)
print("output fields: ", vectordb.fields)


In [ ]:
results = vectordb.similarity_search(query="What is the internet-access log system of VMAX?", k=2)

for i, doc in enumerate(results, 1):
    source = doc.metadata.get("source") or doc.metadata.get("file_path") or "unknown source"
    page = doc.metadata.get("page", "-")
    preview = doc.page_content[:220].replace("\n", " ")
    print(f"===== retrieval result {i} | source={source} | page={page} =====")
    print(preview)
    print()


## Creating an LLM


Here we initialise a chat model against the OpenAI-compatible API exposed by vLLM. For teaching purposes, read it as the "generator": the retriever finds context in the knowledge base, the LLM turns that context into the final answer.

Before running, confirm that the repository root `.env` defines `vLLM_QWEN_BASE_URL`, `vLLM_QWEN_API_KEY` and `vLLM_QWEN_MODEL_ID`.



**OpenAI-compatible interface (vLLM) as the example**


In [ ]:
import os
import httpx
from dotenv import load_dotenv, find_dotenv
from langchain.chat_models import init_chat_model

# load the environment variables
_ = load_dotenv(find_dotenv())
api_key = os.getenv("vLLM_QWEN_API_KEY")
if not api_key:
    raise ValueError("vLLM_QWEN_API_KEY was not found - set it in the .env file at the repository root first.")

# initialise the LLM (vLLM, OpenAI-compatible API)
my_llm = init_chat_model(
    model=os.environ["vLLM_QWEN_MODEL_ID"],
    model_provider="openai",
    base_url=os.environ["vLLM_QWEN_BASE_URL"],
    api_key=api_key,
    http_client=httpx.Client(trust_env=False),
    temperature=0.3,
    max_tokens=3000,
)

result = my_llm.invoke("Who are you?")
print(result.content)


## Building the retrieval QA chain


The prompt is the key control point of the RAG QA chain. This template insists on three things:

- answer from the recalled context only, to reduce improvisation outside the knowledge base;
- if the context has no answer, say "I don't know" explicitly;
- keep the output concise and coherent, suitable for a classroom demo.



In [ ]:
from langchain_core.prompts import PromptTemplate

template = """You are the VMAX operations assistant. Answer strictly from the context below.

context: 
{context}

question: {question}

answer requirements: 
1. answer only from the context and never invent information outside it;
2. if the context does not contain the answer, reply I do not know;
3. keep the answer concise and coherent, without repeating yourself;
4. close with Thank you for your question.

answer: """

QA_CHAIN_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template=template,
)


#### A template-based retrieval chain: basic retrieval version


In [ ]:
from langchain_classic.chains import RetrievalQA

# the baseline starts with plain similarity search for clarity: the question vector -> Milvus Top-K  document chunks.
# to widen the result variety, change search_type to "mmr", and set fetch_k/lambda_mult.
base_retriever = vectordb.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5},
)


In [ ]:
# create the QA chain
qa_chain = RetrievalQA.from_chain_type(
    my_llm,
    retriever=base_retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": QA_CHAIN_PROMPT}
)

#### A template-based retrieval chain: the LCEL version



LCEL (LangChain Expression Language) composes "retrieve documents -> format the context -> fill the prompt -> call the LLM -> parse the output" into one composable chain.


In [ ]:
def format_docs(docs):
    """assemble the retrieved documents into the prompt context, keeping a simple source marker so the evidence stays visible."""
    formatted = []
    for i, doc in enumerate(docs, 1):
        source = doc.metadata.get("source") or doc.metadata.get("file_path") or "unknown source"
        page = doc.metadata.get("page", "-")
        formatted.append(f"[document{i} | source={source} | page={page}]\n{doc.page_content}")
    return "\n\n".join(formatted)


In [ ]:
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

rag_chain = (
    {"context": base_retriever | format_docs, "question": RunnablePassthrough()}
    | QA_CHAIN_PROMPT
    | my_llm
    | StrOutputParser()
)

In [ ]:
question_1 = "What is the 5GC service of ZXVMAX-S?"
result = rag_chain.invoke(question_1)
print("LCELthe retrieval chain answered question_1:")
print(result)

## Testing the retrieval QA chain

Two questions test the baseline RAG chain:

- `question_1` is a specialist question covered by the knowledge base, used to see whether RAG recalls evidence from the VMAX documents and answers from it;
- `question_2` is general knowledge outside the knowledge base, used to see whether the prompt makes the model answer "I don't know" when context is missing.

> **How to read the output**: this project's prompt explicitly demands "answer only I don't know when the context is insufficient", and Qwen3 on vLLM is quite conservative. When the top recalled chunks are table-of-contents or chart pages with low information density, the RAG chain refuses outright while the no-knowledge-base baseline writes much more. That is not a broken pipeline, it is the combined result of **retrieval quality plus a strict-constraint prompt**. In class, check the printed source filename and page number to see whether real body text was hit, then look at the evaluation stage: RAG's advantage shows up in **faithfulness (not inventing content)**, not in answer length. For a more forthcoming answer, increase `k`, switch to hybrid retrieval, or relax the refusal constraint in the prompt.


### A question inside the knowledge base: answering a specialist question

This question goes through the full RAG flow: "embed the question -> Milvus similarity search -> fill the recalled chunks into the prompt -> LLM generates the answer". The code also prints the top 3 source documents so learners can see what the answer is grounded in.



In [ ]:
question_1 = "What is the 5GC service of ZXVMAX-S?/no_think"
result = qa_chain.invoke({"query": question_1})

print("LLM+with the knowledge base, question_1 answered:")
print(result["result"])
print("\nnumber of source documents: ", len(result["source_documents"]))
for i, doc in enumerate(result["source_documents"][:3], 1):
    source = doc.metadata.get("source") or doc.metadata.get("file_path") or "unknown source"
    page = doc.metadata.get("page", "-")
    print(f"source {i}: source={source} | page={page}")


### A question outside the knowledge base: refusal test

This question has nothing to do with the VMAX knowledge base. Ideally the retrieved context cannot support an answer, so the prompt constrains the model to output "I don't know" instead of improvising from general knowledge.



In [ ]:
question_2 = "Who is Jay Chou?/no_think"
result = qa_chain.invoke({"query": question_2})

print("LLM+with the knowledge base, question_2 answered:")
print(result["result"])


### Comparing against the LLM without a knowledge base

To show what RAG contributes, the next cell bypasses the vector database and lets the LLM answer the same questions directly. When comparing, watch for:

- whether the specialist question gets a generic, inaccurate or ungrounded answer;
- whether the out-of-knowledge-base question is answered outright instead of refused.



In [ ]:
prompt_template = f"Please answer the following question:\n{question_1}"

raw_result_1 = my_llm.invoke(prompt_template)
print("without a knowledge base, question_1 answered:")
print(raw_result_1.content)


In [ ]:
prompt_template = f"Please answer the following question:\n{question_2}"

raw_result_2 = my_llm.invoke(prompt_template)
print("without a knowledge base, question_2 answered:")
print(raw_result_2.content)


### Summary of the results

The two questions above show that with a knowledge base attached, the model answers VMAX specialist questions from the local documents first; when the question is unrelated to the knowledge base, the prompt makes it answer "I don't know". Without a knowledge base the model can only rely on its parametric knowledge and easily produces generic or inaccurate answers. That is exactly the core value of RAG: mitigating hallucination and supplying private knowledge.


## Adding conversation memory (optional)

The basic retrieval QA chain handles only the current question. If the user asks something like "now put that in a table" or "simplify the result above", which depends on the previous turn, you need conversation history.

The next cells use `ConversationalRetrievalChain` to demo the simplest multi-turn Q&A. Note that this is the `langchain_classic` teaching API: the current version emits a deprecation warning, but the course examples still run.



In [ ]:
from langchain_classic.memory import ConversationBufferMemory

memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True,
)


### ConversationalRetrievalChain

A conversational retrieval chain first uses the history to rewrite or understand the current question, then retrieves and answers. It suits demos where "the second question depends on the first answer".


In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import ConversationalRetrievalChain

template = """You are the VMAX operations assistant. Answer using the conversation history and context below.

conversation history: 
{chat_history}

relevant context: 
{context}

question: {question}

answer requirements: 
1. answer only from the context and never invent information outside it;
2. if the context does not contain the answer, reply I do not know;
3. close with Thank you for your question.

answer: """

QA_PROMPT = PromptTemplate(
    input_variables=["chat_history", "context", "question"],
    template=template,
)

conversation_qa_chain = ConversationalRetrievalChain.from_llm(
    llm=my_llm,
    retriever=base_retriever,
    memory=memory,
    combine_docs_chain_kwargs={"prompt": QA_PROMPT},
    chain_type="stuff",
)


In [ ]:
questions = [
    "What is the 5GC service of ZXVMAX-S? Which features does it include?/no_think",
    "format the results above as a markdown table/no_think",
]

for question in questions:
    result = conversation_qa_chain.invoke({"question": question})
    print(f"question: {question}")
    print(f"answer: {result['answer']}")
    print("\n" + "=" * 50 + "\n")


### Reading the conversation-chain result

The first question retrieves from the `ZXVMAXS` simple vector store in Milvus and generates an answer about 5GC services. The second question repeats no background and only says "the result above", yet the chain uses the history questions and answers held in `memory` to continue into a table. This shows that memory solves "continuity of multi-turn context" while the vector database solves "retrieval of external knowledge" - two different responsibilities.


# Advanced RAG - hybrid retrieval

The baseline RAG only uses `dense` semantic vector search. It works well for natural-language questions but is not always reliable for literal keywords such as product names, alarm codes, interface names and abbreviations. This chapter uses the `ZXVMAXS_HYBRID` collection built in the previous chapter to demonstrate **dense semantic search + sparse BM25 keyword search**, and then adds reranking on top.

We stay with the VMAX documents for the experiments and watch three things:

1. how hybrid retrieval fuses semantic recall and keyword recall;
2. the difference between the weighted and RRF fusion strategies;
3. how reranking reselects the context that is most suitable to feed into the LLM.



## How hybrid retrieval works

Hybrid retrieval usually has two recall paths:

- **Dense vector search**: encode the query and each document chunk into a dense embedding and recall by semantic similarity - good for natural-language phrasing, paraphrases and vague questions;
- **Sparse / BM25 search**: recall based on keywords, term frequency and inverse document frequency - good for exact matches on product names, module names, alarm codes, interface names, abbreviations and numeric identifiers.

Milvus 2.6 can generate sparse vectors server-side through `BM25BuiltInFunction`. The client only writes `text` and the `dense` embedding, and Milvus derives the `sparse` field from `text` and uses it in hybrid search.

Note: `sparse` is a sparse vector field produced by the BM25 function. It can only be used for scoring, never returned as raw data, so `dense/sparse/vector` must be excluded from `vectorstore.fields` before using the LangChain Milvus retriever.



## Implementing hybrid retrieval: a hand-written version first

Before calling Milvus's built-in hybrid retrieval, let's take "hybrid" apart with some teaching code:

1. **Dense path**: semantic vector search against the simple store `ZXVMAXS`;
2. **BM25 path**: build a local BM25 keyword index over the same document texts;
3. **Score normalisation**: squash both score sets into 0-1;
4. **Weighted fusion**: `hybrid_score = dense_weight * dense_score + bm25_weight * bm25_score`;
5. **Ranked output**: observe how the results change with different weights.

This code is not the recommended production approach; its purpose is to let students see the mechanics underneath hybrid retrieval. Once that is clear, using Milvus's server-side BM25 and ranker feels natural.



### Preparing the teaching dense and BM25 recall paths

BM25 is built from the text field of the `ZXVMAXS` simple retrieval collection. To keep the class fast, at most 500 texts are read; the current sample data has about 322, so all of it is read.



In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())

from pymilvus import MilvusClient
from langchain_milvus import Milvus
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
from rank_bm25 import BM25Okapi
import re

my_emb = OpenAIEmbeddings(
    base_url=os.environ["XINFERENCE_OPENAI_BASE_URL"],
    api_key=os.environ["XINFERENCE_API_KEY"],
    model=os.environ["XINFERENCE_EMBEDDING_MODEL_ID"],
)

connection_args = {
    "uri": os.environ["MILVUS_URI"],
}

_milvus_client = MilvusClient(**connection_args)

# Dense semantic path: use the simple collection ZXVMAXS.
vectordb_dense = Milvus(
    embedding_function=my_emb,
    collection_name="ZXVMAXS",
    connection_args=connection_args,
    vector_field="dense",
    text_field="text",
)
vectordb_dense.fields = [field for field in vectordb_dense.fields if field not in {"dense", "sparse", "vector"}]

# BM25 keyword path: read text/source/page from the same collection and build a local BM25 index.
raw_rows = _milvus_client.query(
    collection_name="ZXVMAXS",
    filter="pk >= 0",
    output_fields=["text", "source", "file_path", "page"],
    limit=500,
)

bm25_docs = [
    Document(
        page_content=row["text"],
        metadata={
            "source": row.get("source") or row.get("file_path") or "unknown source",
            "page": row.get("page", "-"),
        },
    )
    for row in raw_rows
]

def tokenize_zh_en(text: str):
    """teaching tokenizer: split latin words and numbers by token, Chinese by character bigram - enough to demonstrate BM25."""
    text = text.lower()
    tokens = re.findall(r"[a-z0-9][a-z0-9_-]+", text)
    chinese = "".join(re.findall(r"[\u4e00-\u9fff]+", text))
    tokens.extend(chinese[i:i + 2] for i in range(max(len(chinese) - 1, 0)))
    return [token for token in tokens if len(token) >= 2]

bm25_corpus = [tokenize_zh_en(doc.page_content) for doc in bm25_docs]
bm25_index = BM25Okapi(bm25_corpus)

print(f"Dense collection: ZXVMAXS")
print(f"BM25 documents: {len(bm25_docs)}")


### Fixed-weight fusion

Fixed weights are the clearest way to explain "how a hybrid score is composed". Below we set `dense_weight=0.6` and `bm25_weight=0.4`, representing the importance of semantic search and keyword search respectively.



In [ ]:
def normalize_scores(score_map):
    if not score_map:
        return {}
    values = list(score_map.values())
    min_score, max_score = min(values), max(values)
    if max_score == min_score:
        return {key: 1.0 for key in score_map}
    return {key: (value - min_score) / (max_score - min_score) for key, value in score_map.items()}

def doc_key(doc):
    source = str(doc.metadata.get("source") or doc.metadata.get("file_path") or "unknown source").strip()
    page = str(doc.metadata.get("page", "-")).strip()
    return source, page, doc.page_content[:80]

def manual_hybrid_search(query, dense_weight=0.6, bm25_weight=0.4, dense_k=8, bm25_k=8, final_k=5):
    # Dense path: scored retrieval. The direction of the Milvus score depends on the distance metric, so the lesson min-max normalises it and only reads it as a relative ranking.
    dense_pairs = vectordb_dense.similarity_search_with_score(query, k=dense_k)
    dense_docs = [doc for doc, score in dense_pairs]
    dense_raw = {doc_key(doc): float(score) for doc, score in dense_pairs}
    doc_by_key = {doc_key(doc): doc for doc in dense_docs}

    # BM25 path: local keyword search, higher score means more relevant.
    tokenized_query = tokenize_zh_en(query)
    bm25_scores = bm25_index.get_scores(tokenized_query)
    top_bm25_indices = sorted(range(len(bm25_scores)), key=lambda idx: bm25_scores[idx], reverse=True)[:bm25_k]
    bm25_raw = {}
    for idx in top_bm25_indices:
        doc = bm25_docs[idx]
        key = doc_key(doc)
        bm25_raw[key] = float(bm25_scores[idx])
        doc_by_key.setdefault(key, doc)

    dense_norm = normalize_scores(dense_raw)
    bm25_norm = normalize_scores(bm25_raw)

    fused = []
    for key, doc in doc_by_key.items():
        dense_score = dense_norm.get(key, 0.0)
        bm25_score = bm25_norm.get(key, 0.0)
        hybrid_score = dense_weight * dense_score + bm25_weight * bm25_score
        fused.append((hybrid_score, dense_score, bm25_score, doc))

    fused.sort(key=lambda item: item[0], reverse=True)
    return fused[:final_k]

def print_manual_hybrid_results(title, results):
    print(f"\n===== {title} =====")
    for i, (hybrid_score, dense_score, bm25_score, doc) in enumerate(results, 1):
        source = doc.metadata.get("source") or doc.metadata.get("file_path") or "unknown source"
        page = doc.metadata.get("page", "-")
        preview = doc.page_content[:160].replace("\n", " ")
        print(f"{i}. hybrid={hybrid_score:.3f} | dense={dense_score:.3f} | bm25={bm25_score:.3f} | source={source} | page={page}")
        print(f"   {preview}")

manual_query = "ZXVMAX-SWhich features does the internet-access log have?"
fixed_results = manual_hybrid_search(manual_query, dense_weight=0.6, bm25_weight=0.4)
print_manual_hybrid_results("teaching hybrid retrieval with fixed weights", fixed_results)


### Dynamic-weight fusion

Dynamic weighting builds on fixed weights by adjusting the dense/BM25 ratio automatically from the query's characteristics:

- questions with obvious keywords such as product names, alarm codes, interface names or field names get a higher BM25 weight;
- semantic questions asking for causes, optimisation advice or impact analysis get a higher dense weight;
- everything else stays balanced.



In [ ]:
def choose_manual_hybrid_weights(question: str):
    keyword_patterns = [
        r"\d{4,}",
        r"ZXVMAX-S",
        r"interface|field|alarm|log|NAT|IMSI|MSISDN|URL",
        r"which are there|which ones are included|list|table",
    ]
    semantic_patterns = [
        r"why|cause|how to optimise|how to fix it|impact|pros and cons|explain",
    ]
    if any(re.search(pattern, question, re.IGNORECASE) for pattern in keyword_patterns):
        return 0.4, 0.6, "keyword_first"
    if any(re.search(pattern, question, re.IGNORECASE) for pattern in semantic_patterns):
        return 0.7, 0.3, "semantic_first"
    return 0.5, 0.5, "balanced"

for test_query in [
    "ZXVMAX-SWhich features does the internet-access log have?",
    "Why is 5GC service quality fault localisation needed?",
    "114020039How should this alarm be handled?",
]:
    dense_weight, bm25_weight, query_type = choose_manual_hybrid_weights(test_query)
    results = manual_hybrid_search(
        test_query,
        dense_weight=dense_weight,
        bm25_weight=bm25_weight,
        final_k=3,
    )
    print(f"\nQuery: {test_query}")
    print(f"query_type={query_type}, dense_weight={dense_weight}, bm25_weight={bm25_weight}")
    print_manual_hybrid_results("teaching hybrid retrieval with dynamic weights", results)


## Milvus built-in hybrid retrieval + rerank

The hand-written version above teaches the mechanics, but pulling a BM25 index into the client is not recommended in production. Milvus 2.6 generates the sparse field server-side with `BM25BuiltInFunction` and performs weighted or RRF fusion directly at search time.

From here we use the `ZXVMAXS_HYBRID` built in the previous chapter, which is the recommended way to call it in practice.



## Loading the hybrid retrieval vector store

Here we load the `ZXVMAXS_HYBRID` collection created in the previous chapter. It has two vector fields:

- `dense`: the semantic embedding field;
- `sparse`: the keyword retrieval field generated by the Milvus BM25 built-in function.



In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())

from pymilvus import MilvusClient
from langchain_milvus import Milvus, BM25BuiltInFunction
from langchain_openai import OpenAIEmbeddings

my_emb = OpenAIEmbeddings(
    base_url=os.environ["XINFERENCE_OPENAI_BASE_URL"],
    api_key=os.environ["XINFERENCE_API_KEY"],
    model=os.environ["XINFERENCE_EMBEDDING_MODEL_ID"],
)

connection_args = {
    "uri": os.environ["MILVUS_URI"],
}

_milvus_client = MilvusClient(**connection_args)

vectordb_hybrid = Milvus(
    embedding_function=my_emb,
    builtin_function=BM25BuiltInFunction(output_field_names="sparse"),
    collection_name="ZXVMAXS_HYBRID",
    connection_args=connection_args,
    vector_field=["dense", "sparse"],
    text_field="text",
    consistency_level="Bounded",
)

# retrieval returns only text and metadata fields, never the raw dense/sparse vectors.
vectordb_hybrid.fields = [
    field for field in vectordb_hybrid.fields
    if field not in {"dense", "sparse", "vector"}
]

print("vector fields: ", vectordb_hybrid.vector_fields)
print("output fields: ", vectordb_hybrid.fields)


## Weighted hybrid retrieval

The weighted ranker fuses the dense and sparse scores by weight. Below we set `weights=[0.6, 0.4]`:

- `0.6` for dense semantic search;
- `0.4` for sparse BM25 keyword search.

Raise the dense weight for natural-language questions; raise the sparse weight when the question contains keywords such as alarm codes, product names or interface names.



In [ ]:
def print_docs(title, docs, preview_chars=220):
    print(f"\n===== {title} =====")
    displayed = set()
    row = 1
    for doc in docs:
        source = str(doc.metadata.get("source") or doc.metadata.get("file_path") or "unknown source").strip()
        page = str(doc.metadata.get("page", "-")).strip()
        key = (source, page, doc.page_content[:80])
        if key in displayed:
            continue
        displayed.add(key)
        preview = doc.page_content[:preview_chars].replace("\n", " ")
        score = doc.metadata.get("rerank_score")
        score_text = f" | rerank_score={score:.2f}" if isinstance(score, (int, float)) else ""
        print(f"{row}. source={source} | page={page}{score_text}")
        print(f"   {preview}")
        row += 1

query = "ZXVMAX-SWhich features does the internet-access log have?"

weighted_docs = vectordb_hybrid.similarity_search(
    query,
    k=5,
    ranker_type="weighted",
    ranker_params={"weights": [0.6, 0.4]},
)

print_docs("Weighted hybrid retrieval result", weighted_docs)


## RRF hybrid retrieval

RRF (Reciprocal Rank Fusion) needs no manual dense/sparse weights; it fuses purely from the rank positions in each recall list. It is a good fit for teaching and initial experiments: when you are unsure how to set the weights, start from RRF as a robust baseline.



In [ ]:
rrf_docs = vectordb_hybrid.similarity_search(
    query,
    k=5,
    ranker_type="rrf",
    ranker_params={"k": 100},
)

print_docs("RRF hybrid retrieval result", rrf_docs)


## Dynamic weighting strategy

In real business use, different questions want different weights:

- questions containing keywords such as alarm codes, interface names, product names, field names or identifiers suit a higher BM25 sparse weight;
- questions phrased naturally that need semantic understanding suit a higher dense weight;
- when unsure, use balanced weights or go straight to RRF.

The next cell demonstrates dynamic weighting with a simple rule. It is not a final production design, but it helps students understand why the weights matter.



In [ ]:
import re

def choose_hybrid_weights(question: str):
    keyword_patterns = [
        r"\d{4,}",          # long numbers, alarm codes, identifiers
        r"ZXVMAX-S",          # product name
        r"interface|field|alarm|log|NAT|IMSI|MSISDN|URL",
        r"which are there|which ones are included|list|table",
    ]
    semantic_patterns = [
        r"why|cause|how to optimise|how to fix it|impact|pros and cons|explain",
    ]

    if any(re.search(pattern, question, re.IGNORECASE) for pattern in keyword_patterns):
        return [0.4, 0.6], "keyword_first"
    if any(re.search(pattern, question, re.IGNORECASE) for pattern in semantic_patterns):
        return [0.7, 0.3], "semantic_first"
    return [0.5, 0.5], "balanced"

for test_query in [
    "ZXVMAX-SWhich features does the internet-access log have?",
    "Why is 5GC service quality fault localisation needed?",
    "114020039How should this alarm be handled?",
]:
    weights, query_type = choose_hybrid_weights(test_query)
    docs = vectordb_hybrid.similarity_search(
        test_query,
        k=3,
        ranker_type="weighted",
        ranker_params={"weights": weights},
    )
    print(f"\nQuery: {test_query}")
    print(f"query_type={query_type}, weights={weights}")
    print_docs("dynamic-weight retrieval result", docs, preview_chars=120)


## Reranking

Hybrid retrieval solves "recall as many plausibly relevant candidates as possible", but the candidate list still contains redundant or weakly relevant content. A reranker scores each (query, candidate) pair and reorders, which is usually used as:

1. recall more candidates with hybrid search first, say top 10 or top 20;
2. rerank them and keep the top 3 to top 5;
3. hand only the reranked high-quality context to the LLM.

This section shows three rerank options:

- **Cohere Rerank**: a mature cloud API, requires `COHERE_API_KEY`;
- **Jina Reranker**: a cloud HTTP API with multilingual support, requires `JINA_API_KEY`;
- **Qwen3 Reranker (used by this project)**: called through Xinference's `XINFERENCE_RERANKER_MODEL_ID` on the OpenAI-compatible `/v1/rerank` endpoint, privately deployed, no external API key needed.



### Cohere Rerank (example only)

If `langchain-cohere` is installed and `COHERE_API_KEY` is configured, you can use `langchain_cohere.CohereRerank`. Just skip this cell when the dependency or the API key is missing.



In [ ]:
import os

# this needs both the langchain-cohere dependency and COHERE_API_KEY, otherwise the example is skipped.
if os.getenv("COHERE_API_KEY"):
    try:
        from langchain_cohere import CohereRerank
    except ImportError:
        print("langchain-cohere is not installed, skipping the Cohere Rerank example. Run uv add langchain-cohere and try again.")
    else:
        cohere_compressor = CohereRerank(
            model="rerank-multilingual-v3.0",
            top_n=3,
        )
        cohere_reranked_docs = cohere_compressor.compress_documents(weighted_docs, query=query)
        print_docs("Cohere Rerank result", cohere_reranked_docs)
else:
    print("COHERE_API_KEY is not configured, skipping the Cohere Rerank example.")


### Jina Reranker (example only)

Jina Reranker can be called straight over its HTTP API. The generic template is below: if `JINA_API_KEY` is configured it performs a real call, otherwise it only prints a skip message.



In [ ]:
import os
import httpx
from langchain_core.documents import Document

def jina_rerank(query, documents, top_n=3, model="jina-reranker-v2-base-multilingual"):
    api_key = os.getenv("JINA_API_KEY")
    if not api_key:
        print("JINA_API_KEY is not configured, skipping the Jina Rerank example.")
        return []

    payload = {
        "model": model,
        "query": query,
        "top_n": top_n,
        "documents": [doc.page_content for doc in documents],
    }
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    }
    response = httpx.post(
        "https://api.jina.ai/v1/rerank",
        headers=headers,
        json=payload,
        timeout=60,
        trust_env=False,
    )
    response.raise_for_status()
    results = response.json().get("results", [])

    reranked = []
    for item in results:
        idx = item["index"]
        score = item.get("relevance_score", 0.0)
        doc = documents[idx]
        reranked.append(Document(
            page_content=doc.page_content,
            metadata={**doc.metadata, "rerank_score": float(score), "rerank_model": model},
        ))
    return reranked

jina_reranked_docs = jina_rerank(query, weighted_docs, top_n=3)
if jina_reranked_docs:
    print_docs("Jina Rerank result", jina_reranked_docs)


The next cell wraps `/v1/rerank` into `xinference_rerank(query, documents, top_n)`, which returns documents carrying a `rerank_score` metadata field and can drop into any "recall then rerank" position.

> **How to read the output**: the Qwen3 Reranker's relevance scores saturate noticeably towards `1.0`, so in class focus on **whether the ordering is sensible** and never treat the absolute score as a threshold.


In [ ]:
import os
import httpx
from dotenv import load_dotenv, find_dotenv
from langchain_core.documents import Document

_ = load_dotenv(find_dotenv())

# prefer the reranker settings from the unified configuration cell, otherwise read .env
RERANKER_BASE_URL = globals().get("RERANKER_BASE_URL") or os.environ["XINFERENCE_OPENAI_BASE_URL"]
RERANKER_MODEL = globals().get("RERANKER_MODEL") or os.environ["XINFERENCE_RERANKER_MODEL_ID"]


def xinference_rerank(query, documents, top_n=3):
    """call Xinference /v1/rerank to precisely rank the candidates and return them in descending relevance."""
    if not documents:
        return []

    response = httpx.post(
        RERANKER_BASE_URL.rstrip("/") + "/rerank",
        headers={"Authorization": "Bearer " + os.environ["XINFERENCE_API_KEY"]},
        json={
            "model": RERANKER_MODEL,
            "query": query,
            "documents": [doc.page_content for doc in documents],
            "top_n": min(top_n, len(documents)),
        },
        timeout=60,
        trust_env=False,
    )
    response.raise_for_status()

    scored_docs = []
    for item in response.json().get("results", []):
        doc = documents[item["index"]]
        scored_docs.append(Document(
            page_content=doc.page_content,
            metadata={
                **doc.metadata,
                "rerank_score": float(item.get("relevance_score", 0.0)),
                "rerank_model": RERANKER_MODEL,
            },
        ))

    scored_docs.sort(key=lambda item: item.metadata.get("rerank_score", 0.0), reverse=True)
    return scored_docs


## Hybrid retrieval + local rerank in practice

The next cell recalls candidates with weighted hybrid retrieval, then reranks them with the Qwen3 Reranker on Xinference and keeps 3. In class, compare the source pages and content snippets before and after reranking to see whether reranking pushed the most relevant context to the front.



In [ ]:
rerank_query = "ZXVMAX-SWhich features does the internet-access log have?"

candidate_docs = vectordb_hybrid.similarity_search(
    rerank_query,
    k=3,
    ranker_type="weighted",
    ranker_params={"weights": [0.6, 0.4]},
)

print_docs("Rerank before: hybrid candidates", candidate_docs, preview_chars=140)

reranked_docs = xinference_rerank(rerank_query, candidate_docs, top_n=3)
print_docs("Rerank after: Xinference Qwen3 reranked", reranked_docs, preview_chars=180)


## Chapter summary

The recommended practice path in this chapter is:

1. **Baseline**: get a working RAG with dense vector search first;
2. **Advanced**: add the Milvus BM25 sparse field and do hybrid retrieval with weighted or RRF fusion;
3. **Quality**: when recall is noisy, add reranking after hybrid retrieval so the most relevant context rises to the top;
4. **Model choice**: Cohere/Jina suit cloud API scenarios, the Qwen3 Reranker on Xinference suits intranets, private deployments and teaching environments.

One thing to keep in mind: Milvus's `sparse` field is a retrieval field, not a display field. Whether or not you rerank, what you hand back to LangChain document objects should be ordinary fields such as `text/source/page`.



## The complete hybrid retrieval QA chain (with memory)

Hybrid retrieval and reranking above solve "recall and ranking", but a complete RAG teaching case also has to wire them into a real QA chain.

This section adds the missing three pieces:

1. **Retrieval**: keep using weighted hybrid retrieval on `ZXVMAXS_HYBRID`;
2. **Generation**: a single answer template that organises context, question and answer requirements;
3. **Memory**: `RunnableWithMessageHistory` stores the multi-turn conversation so follow-ups can rely on what came before.



### Why this chain is more complete

Compared with the single-turn retrieval QA above, this adds a "conversation state":

- Turn 1 asks "What features does the ZXVMAX-S internet-access log have?", and the system retrieves first, then generates;
- Turn 2 asks "And how long can it be stored?", and the system carries the previous turn into the answer so the pronoun "it" is not misread;
- To show the internals in class, print the history and the rewritten retrieval query directly.

This is much closer to a continuous Q&A assistant in a real business setting.



In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables import RunnableLambda
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

_ = load_dotenv(find_dotenv())

# reuse the hybrid store created in an earlier section; if it is missing in this session, rebuild it from the unified configuration.
qa_hybrid_db = globals().get("vectordb_hybrid")
if qa_hybrid_db is None:
    from pymilvus import MilvusClient
    from langchain_milvus import Milvus, BM25BuiltInFunction
    from langchain_openai import OpenAIEmbeddings

    qa_connection_args = {
        "uri": os.environ["MILVUS_URI"],
    }
    qa_emb = OpenAIEmbeddings(
        base_url=os.environ["XINFERENCE_OPENAI_BASE_URL"],
        api_key=os.environ["XINFERENCE_API_KEY"],
        model=os.environ["XINFERENCE_EMBEDDING_MODEL_ID"],
    )
    _qa_milvus_client = MilvusClient(**qa_connection_args)

    qa_hybrid_db = Milvus(
        embedding_function=qa_emb,
        builtin_function=BM25BuiltInFunction(output_field_names="sparse"),
        collection_name="ZXVMAXS_HYBRID",
        connection_args=qa_connection_args,
        vector_field=["dense", "sparse"],
        text_field="text",
        consistency_level="Bounded",
    )

# reuse the hybrid store built earlier, and filter the fields again so sparse/dense is never returned as raw data.
qa_hybrid_db.fields = [
    field for field in qa_hybrid_db.fields
    if field not in {"dense", "sparse", "vector"}
]

# reuse the LLM created in an earlier section; if it is missing in this session, rebuild it from the unified model configuration.
qa_llm = globals().get("my_llm")
if qa_llm is None:
    import os
    import httpx
    from dotenv import load_dotenv, find_dotenv
    from langchain.chat_models import init_chat_model

    _ = load_dotenv(find_dotenv())
    api_key = os.getenv("vLLM_QWEN_API_KEY")
    if not api_key:
        raise RuntimeError("vLLM_QWEN_API_KEY is not configured, so the retrieval QA chain example cannot run.")

    qa_llm = init_chat_model(
        model=os.environ["vLLM_QWEN_MODEL_ID"],
        model_provider="openai",
        base_url=os.environ["vLLM_QWEN_BASE_URL"],
        api_key=api_key,
        http_client=httpx.Client(trust_env=False),
        temperature=0.7,
        max_tokens=10000,
    )

rewrite_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a retrieval query rewriter. Using the conversation history, rewrite the current question into a standalone, complete question suited to retrieval. Output only the rewritten question, no explanation.",
    ),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}"),
])

answer_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are the VMAX operations assistant. Answer only from the given context, never invent anything."
        "if the context does not contain the answer, answer I do not know explicitly."
        "append the cited sources and page numbers at the end of the answer.",
    ),
    MessagesPlaceholder(variable_name="chat_history"),
    (
        "human",
        "original question: {question}\n\n"
        "rewritten retrieval question: {standalone_question}\n\n"
        "relevant context: \n{context}\n\n"
        "answer briefly and accurately.",
    ),
])

question_rewriter = rewrite_prompt | qa_llm | StrOutputParser()


def _format_retrieval_docs(docs, preview_chars=260):
    lines = []
    for i, doc in enumerate(docs, 1):
        source = str(doc.metadata.get("source") or doc.metadata.get("file_path") or "unknown source")
        page = str(doc.metadata.get("page", "-"))
        preview = doc.page_content[:preview_chars].replace("\n", " ")
        lines.append(f"[{i}] source={source} | page={page}\n{preview}")
    return "\n\n".join(lines)


def build_hybrid_qa_inputs(inputs: dict):
    question = inputs["question"]
    chat_history = inputs.get("chat_history", [])

    standalone_question = question
    if chat_history:
        standalone_question = question_rewriter.invoke({
            "question": question,
            "chat_history": chat_history,
        }).strip()

    docs = qa_hybrid_db.similarity_search(
        standalone_question,
        k=6,
        ranker_type="weighted",
        ranker_params={"weights": [0.6, 0.4]},
    )

    if "xinference_rerank" in globals():
        docs = xinference_rerank(standalone_question, docs, top_n=3)

    return {
        "question": question,
        "standalone_question": standalone_question,
        "chat_history": chat_history,
        "context": _format_retrieval_docs(docs),
    }

hybrid_qa_chain = (
    RunnableLambda(build_hybrid_qa_inputs)
    | answer_prompt
    | qa_llm
    | StrOutputParser()
)

conversation_store = {}


def get_hybrid_qa_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in conversation_store:
        conversation_store[session_id] = ChatMessageHistory()
    return conversation_store[session_id]

hybrid_qa_with_memory = RunnableWithMessageHistory(
    hybrid_qa_chain,
    get_hybrid_qa_history,
    input_messages_key="question",
    history_messages_key="chat_history",
)

print("the hybrid retrieval QA chain is ready.")
print("- supports weighted hybrid retrieval")
print("- supports the optional Xinference Rerank")
print("- conversation memory supported")


### Single-turn first

Run one typical feature question first, to confirm the complete QA chain works out of the box.



In [ ]:
qa_session_id = "hybrid_qa_demo"
first_question = "ZXVMAX-SWhich features does the internet-access log have?"
first_answer = hybrid_qa_with_memory.invoke(
    {"question": first_question},
    config={"configurable": {"session_id": qa_session_id}},
)

print("user question: ", first_question)
print("\nmodel answer: \n")
print(first_answer)

### Then a follow-up that needs memory

The second question deliberately omits the full subject, so we can see whether the system uses the previous turn to complete it before retrieving.



In [ ]:
follow_up_question = "How long does it support keeping them?"

# pull the history of the current session first, and show the rewritten retrieval question on the way.
current_history = get_hybrid_qa_history(qa_session_id).messages
preview_inputs = {
    "question": follow_up_question,
    "chat_history": current_history,
}
preview_result = build_hybrid_qa_inputs(preview_inputs)

print("user follow-up: ", follow_up_question)
print("rewritten retrieval question: ", preview_result["standalone_question"])
print("\nretrieval context preview: \n")
print(preview_result["context"][:1400])

second_answer = hybrid_qa_with_memory.invoke(
    {"question": follow_up_question},
    config={"configurable": {"session_id": qa_session_id}},
)

print("\nmodel answer: \n")
print(second_answer)

### The teaching value of this version

This section turns "hybrid retrieval" into a genuinely conversational RAG chain:

- the retrieval layer handles recall and ranking;
- the rewriting layer turns follow-ups into retrievable questions;
- the memory layer keeps the context;
- the generation layer organises retrieved content into a stable answer.

At this point the hybrid retrieval chapter is more than a retrieval comparison: it covers the full "recall - rerank - generate - memory" loop.


# Advanced RAG - Agentic RAG

The baseline RAG and the hybrid retrieval RAG so far are both "developer-fixed pipelines": every user question must enter one particular retrieval chain, and the LLM answers from its results.

Agentic RAG instead wraps retrieval as tools and lets an agent decide per question:

```text
user question -> agent classifies the task -> picks a tool -> observes the tool result -> produces the final answer
```

This is the same agent idea as the agent chapter of the course, only here ordinary tools are upgraded into RAG tools:

- **simple retrieval tool**: based on `ZXVMAXS`, dense semantic retrieval only, best for semantic questions;
- **hybrid retrieval tool**: based on `ZXVMAXS_HYBRID`, dense + BM25 fused, best for questions heavy in product names, module names, alarm codes and proper nouns;
- **calculation tool**: demonstrates that an agent is not limited to the knowledge base and may pick another tool to finish the job.

Read it this way in class: RAG is one capability, the agent is the decision layer that schedules these capabilities.


## Designing the Agentic RAG tools

Good tool design is not about "pushing all the logic onto the agent"; it is about drawing each tool's boundary clearly:

| Tool | Best for | Under the hood |
|---|---|---|
| `simple_rag_search` | Semantically phrased questions, concept explanations | Milvus dense vector search |
| `hybrid_rag_search` | Questions with obvious keywords: product names, module names, alarm codes, interface names, field names | Milvus dense + BM25 hybrid search, rerank optional |
| `calculate` | Numeric and simple expression arithmetic | Python AST allowlisted evaluation |

A tool's docstring goes into the model context and directly determines whether the agent picks the right tool. Keep tool descriptions specific, restrained and distinguishable.


## Preparing the Agentic RAG tools

The code below reconnects to the two Milvus collections already built, so this section can run standalone:

- `ZXVMAXS`: the simple dense retrieval collection;
- `ZXVMAXS_HYBRID`: the hybrid collection, with a `dense` and a `sparse` vector field.

Reminder: Milvus vector fields exist for retrieval only and must not be returned as raw fields, so the code still filters out `dense/sparse/vector`.


In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())

from pymilvus import MilvusClient
from langchain_milvus import Milvus, BM25BuiltInFunction
from langchain_openai import OpenAIEmbeddings
from langchain_core.tools import tool

# this section rebuilds its own connection so the Agentic RAG chapter can run on its own.
agent_connection_args = {
    "uri": os.environ["MILVUS_URI"],
}

agent_emb = OpenAIEmbeddings(
    base_url=os.environ["XINFERENCE_OPENAI_BASE_URL"],
    api_key=os.environ["XINFERENCE_API_KEY"],
    model=os.environ["XINFERENCE_EMBEDDING_MODEL_ID"],
)

_agent_milvus_client = MilvusClient(**agent_connection_args)

agent_simple_db = Milvus(
    embedding_function=agent_emb,
    collection_name="ZXVMAXS",
    connection_args=agent_connection_args,
    vector_field="dense",
    text_field="text",
    consistency_level="Bounded",
)
agent_simple_db.fields = [
    field for field in agent_simple_db.fields
    if field not in {"dense", "sparse", "vector"}
]

agent_hybrid_db = Milvus(
    embedding_function=agent_emb,
    builtin_function=BM25BuiltInFunction(output_field_names="sparse"),
    collection_name="ZXVMAXS_HYBRID",
    connection_args=agent_connection_args,
    vector_field=["dense", "sparse"],
    text_field="text",
    consistency_level="Bounded",
)
agent_hybrid_db.fields = [
    field for field in agent_hybrid_db.fields
    if field not in {"dense", "sparse", "vector"}
]

def _format_tool_docs(docs, max_chars=320):
    """compress the retrieval result into text that a tool can hand back to the agent."""
    lines = []
    for i, doc in enumerate(docs, 1):
        source = doc.metadata.get("source") or doc.metadata.get("file_path") or "unknown source"
        page = doc.metadata.get("page", "-")
        content = doc.page_content[:max_chars].replace("\n", " ")
        lines.append(f"[{i}] source={source} | page={page}\n{content}")
    return "\n\n".join(lines)

@tool
def simple_rag_search(question: str) -> str:
    """Search the ZXVMAX-S knowledge base with plain dense vector retrieval. Best for semantic, descriptive and conceptual questions."""
    docs = agent_simple_db.similarity_search(question, k=4)
    if not docs:
        return "simple retrieval found no relevant material."
    return _format_tool_docs(docs)

@tool
def hybrid_rag_search(question: str) -> str:
    """Search the ZXVMAX-S knowledge base with Milvus hybrid retrieval. Best for questions containing keywords such as product names, module names, alarm codes, interface names or field names."""
    docs = agent_hybrid_db.similarity_search(
        question,
        k=4,
        ranker_type="weighted",
        ranker_params={"weights": [0.5, 0.5]},
    )
    if "xinference_rerank" in globals():
        docs = xinference_rerank(question, docs, top_n=4)
    if not docs:
        return "hybrid retrieval found no relevant material."
    return _format_tool_docs(docs)

@tool
def calculate(expression: str) -> str:
    """Evaluate a math expression safely. Supports the four arithmetic operations, exponentiation and parenthesised expressions. Do not use it for knowledge-base questions."""
    import ast
    import operator

    operators = {
        ast.Add: operator.add,
        ast.Sub: operator.sub,
        ast.Mult: operator.mul,
        ast.Div: operator.truediv,
        ast.Pow: operator.pow,
        ast.USub: operator.neg,
    }

    def safe_eval(node):
        if isinstance(node, ast.Expression):
            return safe_eval(node.body)
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in operators:
            return operators[type(node.op)](safe_eval(node.left), safe_eval(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in operators:
            return operators[type(node.op)](safe_eval(node.operand))
        raise ValueError(f"unsupported expression: {ast.dump(node)}")

    try:
        tree = ast.parse(expression, mode="eval")
        return f"calculation result: {expression} = {safe_eval(tree)}"
    except Exception as exc:
        return f"calculation error: {exc}"

agentic_rag_tools = [simple_rag_search, hybrid_rag_search, calculate]

print("Agentic RAG tools are ready: ")
for item in agentic_rag_tools:
    print("-", item.name, ":", item.description)

## Test the tools directly first

Before handing them to the agent, call the tools themselves. That has two benefits:

1. it confirms the tool and its external system connect properly;
2. students see exactly what a tool returns before "the agent calls a tool".

If the tool itself is broken, no amount of agent cleverness produces the right result.


In [ ]:
tool_test_questions = [
    (simple_rag_search, {"question": "Why is 5GC service quality fault localisation needed?"}),
    (hybrid_rag_search, {"question": "ZXVMAX-SWhich features does the internet-access log have?"}),
    (calculate, {"expression": "(15 * 37 + 228) / 3"}),
]

for tool_item, args in tool_test_questions:
    print("=" * 80)
    print("tool: ", tool_item.name)
    print("input: ", args)
    print(tool_item.invoke(args)[:1200])

## Building the tool-calling agent

This environment uses LangChain 1.x, where the recommended way to build an agent graph is `create_agent()`; the `AgentExecutor` and `create_tool_calling_agent` seen in older tutorials are no longer exported from `langchain.agents` in some newer versions.

The key difference from an ordinary RAG chain:

- ordinary RAG chain: the developer decided upfront which retriever is always used;
- Agentic RAG: the model reads the question first, then decides whether to call a tool, which one, and how to use the result.

`debug` is off by default so the output stays readable as course material. To watch model messages, tool calls and tool results in class, change `DEBUG_AGENTIC_RAG` in the code to `True`.


In [ ]:
import os
import httpx
from dotenv import load_dotenv, find_dotenv
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent

_ = load_dotenv(find_dotenv())
agent_api_key = os.getenv("vLLM_QWEN_API_KEY")

agentic_system_prompt = (
    "You are an enterprise knowledge-base question answering assistant."
    "you may use the tools to query the ZXVMAX-S knowledge base or to do arithmetic."
    "when the question contains explicit keywords such as a product name, module name, alarm code, interface name or field name, prefer hybrid_rag_search."
    "when the question is closer to a semantic description, a reason or a concept, use simple_rag_search."
    "when the question is arithmetic, use calculate."
    "the answer must rest on what the tools returned; if a tool gave no evidence, state clearly that the knowledge base did not provide enough evidence."
    "the knowledge-base answer must end with its citations and page numbers."
)

if not agent_api_key:
    print("vLLM_QWEN_API_KEY is not configured, skipping the agent model call; the direct tool test above is still enough to understand Agentic RAG.")
else:
    agentic_llm = init_chat_model(
        model=os.environ["vLLM_QWEN_MODEL_ID"],
        model_provider="openai",
        base_url=os.environ["vLLM_QWEN_BASE_URL"],
        api_key=agent_api_key,
        http_client=httpx.Client(trust_env=False),
        temperature=0,
        max_tokens=4000,
    )

    DEBUG_AGENTIC_RAG = False

    agentic_agent = create_agent(
        model=agentic_llm,
        tools=agentic_rag_tools,
        system_prompt=agentic_system_prompt,
        debug=DEBUG_AGENTIC_RAG,
    )

    print("Agentic RAG Agent was created.")

## Testing Agentic RAG

Three kinds of question check whether the agent picks the right tool:

1. **keyword-heavy question**: should prefer hybrid retrieval;
2. **semantic explanation question**: may use simple retrieval;
3. **calculation question**: should use the calculation tool rather than knowledge-base retrieval.

In a real project, never judge an agent's tool selection from a single demo; keep observing with LangSmith, logs or an evaluation set.


In [ ]:
agentic_examples = [
    "ZXVMAX-SWhich features does the internet-access log have?",
    "Why is 5GC service quality fault localisation needed?",
    "please calculate (15 * 37 + 228) / 3",
]

def print_agent_answer(result):
    """compatible with the messages structure returned by LangChain 1.x create_agent."""
    messages = result.get("messages", []) if isinstance(result, dict) else []
    if not messages:
        print(result)
        return
    final_message = messages[-1]
    content = getattr(final_message, "content", final_message)
    print(content)

if "agentic_agent" not in globals():
    print("Agentic Agent has not been created, usually because vLLM_QWEN_API_KEY is not configured.")
else:
    for question in agentic_examples:
        print("=" * 80)
        print("user question: ", question)
        response = agentic_agent.invoke({
            "messages": [{"role": "user", "content": question}],
        })
        print("final answer: ")
        print_agent_answer(response)

## Agentic RAG summary

Agentic RAG does not replace ordinary RAG; it adds a "tool scheduling layer" once ordinary RAG is stable. Suggested teaching and rollout order:

1. **ordinary RAG first**: keep retrieval, prompts and answer quality under control;
2. **then hybrid retrieval + rerank**: improve recall and ranking quality;
3. **then the agent**: wrap retrieval chains, calculation tools and business APIs as tools and let the model choose per task;
4. **keep evaluating tool selection**: the core risk of an agent is not whether the code runs, but whether it chooses the right tool in the right situation.

In one sentence: ordinary RAG answers "how to answer from a knowledge base", Agentic RAG answers "when to search, where to search, and whether another tool is needed".


# Evaluating the RAG system


Once the RAG pipeline is built, the last step is to check whether it is actually more reliable than a bare LLM.

This chapter uses the **LLM-as-Judge** idea for a lightweight comparison:

1. prepare 3 test questions in the VMAX domain, each with reference answer points;
2. generate answers twice: with the **hybrid retrieval QA chain**, and by **calling the LLM directly (no knowledge base)**;
3. have another LLM score both from the **faithfulness, relevance and completeness** angles (1-5, 5 best);
4. compare the two in a table and observe how RAG suppresses hallucination and improves factual accuracy.

> Note: no extra evaluation library (such as Ragas) is introduced here; the already-configured LLM acts as the judge, so this runs directly in the environment. Treat the scores as indicative; real projects should add human annotation.


In [ ]:
# evaluation setup: reuse the unified model configuration and connect to the hybrid vector store
import os
import httpx
import re
import pandas as pd
from dotenv import load_dotenv, find_dotenv
from langchain.chat_models import init_chat_model
from langchain_openai import OpenAIEmbeddings
from langchain_milvus import Milvus, BM25BuiltInFunction
from pymilvus import MilvusClient
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

_ = load_dotenv(find_dotenv())
api_key = os.getenv("vLLM_QWEN_API_KEY")
if not api_key:
    raise ValueError("please configure vLLM_QWEN_API_KEY")

# judge/generation model: temperature 0 keeps the scores stable
eval_llm = init_chat_model(
    model=os.environ["vLLM_QWEN_MODEL_ID"],
    model_provider="openai",
    base_url=os.environ["vLLM_QWEN_BASE_URL"],
    api_key=api_key,
    http_client=httpx.Client(trust_env=False),
    temperature=0,
    max_tokens=2000,
)

eval_emb = OpenAIEmbeddings(
    base_url=os.environ["XINFERENCE_OPENAI_BASE_URL"],
    api_key=os.environ["XINFERENCE_API_KEY"],
    model=os.environ["XINFERENCE_EMBEDDING_MODEL_ID"],
)

connection_args = {"uri": os.environ["MILVUS_URI"]}
_eval_milvus_client = MilvusClient(**connection_args)

eval_hybrid_db = Milvus(
    embedding_function=eval_emb,
    builtin_function=BM25BuiltInFunction(output_field_names="sparse"),
    collection_name="ZXVMAXS_HYBRID",
    connection_args=connection_args,
    vector_field=["dense", "sparse"],
    text_field="text",
    consistency_level="Bounded",
)
# do not return raw vector fields when searching
eval_hybrid_db.fields = [
    field for field in eval_hybrid_db.fields
    if field not in {"dense", "sparse", "vector"}
]

def _format_eval_context(docs, preview_chars=300):
    lines = []
    for i, doc in enumerate(docs, 1):
        source = doc.metadata.get("source") or doc.metadata.get("file_path") or "unknown source"
        page = doc.metadata.get("page", "-")
        content = doc.page_content[:preview_chars].replace("\n", " ")
        lines.append(f"[{i}] source={source} | page={page}\n{content}")
    return "\n\n".join(lines)

def retrieve_for_eval(question: str, k: int = 5):
    docs = eval_hybrid_db.similarity_search(
        question,
        k=k,
        ranker_type="weighted",
        ranker_params={"weights": [0.6, 0.4]},
    )
    return _format_eval_context(docs)

print("evaluation setup done.")


In [ ]:
# build the RAG chain and the baseline chain
rag_prompt = PromptTemplate.from_template("""
You are the VMAX operations assistant. Answer strictly from the context below.

context: 
{context}

question: {question}

answer requirements: 
1. answer only from the context and never invent information outside it;
2. if the context does not contain the answer, reply I do not know;
3. answer briefly and accurately.

answer: """)

rag_answer_chain = (
    {"context": lambda q: retrieve_for_eval(q), "question": RunnablePassthrough()}
    | rag_prompt
    | eval_llm
    | StrOutputParser()
)

baseline_answer_chain = eval_llm | StrOutputParser()

# evaluation set: question + reference key points
eval_set = [
    {
        "question": "ZXVMAX-SWhich features does the internet-access log have?",
        "reference": "internet-access log storage, log query, bulk import query, log management, account/role management, resource monitoring, alarm management, provincial gateway integration, automatic probe-result comparison, NAT log ingestion, northbound interface, historical log query, cloudified XDR query",
    },
    {
        "question": "What is the 5GC service of ZXVMAX-S?",
        "reference": "a multi-dimensional value analytics system for the 5G core network, covering 5G precise provisioning, 5G service operations, one-click complaint handling, area awareness, user experience awareness, real-time and session tracing",
    },
    {
        "question": "114020039How should this alarm be handled?",
        "reference": "NodeManagerprocess local disk usage alarm; likely cause is Yarn usage above the threshold; steps are to raise the alarm threshold, expand the disk, collect logs and contact support",
    },
]

print(f"prepare evaluation {len(eval_set)}  questions")


In [ ]:
# generate the answers (this calls the LLM and Milvus and can take tens of seconds)
for item in eval_set:
    item["rag_answer"] = rag_answer_chain.invoke(item["question"])
    item["baseline_answer"] = baseline_answer_chain.invoke(item["question"])

print("RAG and the baseline answers were generated.")


In [ ]:
# LLM-as-Judge: score the answers on three dimensions
judge_prompt = PromptTemplate.from_template("""
You are an impartial evaluation expert. Grade the assistant answer below against the context, the question and the key points of a reference answer.

[context]
{context}

[question]
{question}

[reference key points]
{reference}

[assistant answer]
{answer}

score the answer on the three dimensions below (1-5, 5 is best) and give a short reason:
1. faithfulness: is every statement supported by the context?
2. relevance: does the answer address the question instead of talking past it?
3. completeness: does the answer cover the reference key points?

follow this output format exactly:
faithfulness: <score>
relevance: <score>
completeness: <score>
reason: <a one-sentence reason>
""")

judge_chain = judge_prompt | eval_llm | StrOutputParser()

def parse_judge_result(text: str):
    result = {"faithfulness": 0, "relevance": 0, "completeness": 0, "reason": ""}
    faith = re.search("faithfulness[:\uff1a]\s*(\d+)", text)
    rel = re.search("relevance[:\uff1a]\s*(\d+)", text)
    comp = re.search("completeness[:\uff1a]\s*(\d+)", text)
    reason = re.search("reason[:\uff1a]\s*(.+)", text, re.DOTALL)
    if faith:
        result["faithfulness"] = int(faith.group(1))
    if rel:
        result["relevance"] = int(rel.group(1))
    if comp:
        result["completeness"] = int(comp.group(1))
    if reason:
        result["reason"] = reason.group(1).strip().replace("\n", " ")
    return result

for item in eval_set:
    context = retrieve_for_eval(item["question"])
    for key in ["rag_answer", "baseline_answer"]:
        judge_text = judge_chain.invoke({
            "context": context,
            "question": item["question"],
            "reference": item["reference"],
            "answer": item[key],
        })
        item[f"{key}_score"] = parse_judge_result(judge_text)

print("scoring done.")


In [ ]:
# summarise the comparison
rows = []
for item in eval_set:
    rows.append({
        "question": item["question"],
        "solution": "RAG",
        "faithfulness": item["rag_answer_score"]["faithfulness"],
        "relevance": item["rag_answer_score"]["relevance"],
        "completeness": item["rag_answer_score"]["completeness"],
        "average": round(sum([item["rag_answer_score"][k] for k in ["faithfulness", "relevance", "completeness"]]) / 3, 2),
    })
    rows.append({
        "question": item["question"],
        "solution": "baseline (no RAG)",
        "faithfulness": item["baseline_answer_score"]["faithfulness"],
        "relevance": item["baseline_answer_score"]["relevance"],
        "completeness": item["baseline_answer_score"]["completeness"],
        "average": round(sum([item["baseline_answer_score"][k] for k in ["faithfulness", "relevance", "completeness"]]) / 3, 2),
    })

df = pd.DataFrame(rows)
display(df)

print("\n===== average score comparison =====")
print(df.groupby("solution")[["faithfulness", "relevance", "completeness", "average"]].mean().round(2))

print("\n===== a detailed reason example (first question) =====")
print("question: ", eval_set[0]["question"])
print("\n[RAG answer]")
print(eval_set[0]["rag_answer"][:300])
print("reason: ", eval_set[0]["rag_answer_score"]["reason"])
print("\n[baseline answer]")
print(eval_set[0]["baseline_answer"][:300])
print("reason: ", eval_set[0]["baseline_answer_score"]["reason"])


## Reading the evaluation result

From this simple LLM-as-Judge experiment you will typically observe:

- **RAG**: faithfulness and completeness are usually clearly higher than the baseline, because answers come from the local knowledge base;
- **baseline**: often produces answers that "sound professional but are invented" (for instance describing ZXVMAX-S as a Huawei product), so hallucination risk is higher;
- **relevance**: with a clearly phrased question both are usually fine; but when the question needs specific keywords from the documents, RAG is steadier.

## Limitations and next optimisations

1. **LLM judges are biased too**: the scoring model may prefer longer or "more complete" answers, so results are indicative only;
2. **Tiny sample**: only 3 questions here; real projects should prepare an evaluation set of 50+;
3. **Retrieval quality is not measured**: add retrieval metrics such as `Recall@K` and `MRR`;
4. **Bring in Ragas**: for something more systematic, install `ragas` and use its built-in `faithfulness`, `answer_relevancy`, `context_recall` metrics.

This chapter moves RAG from "it runs" to "it can be measured", closing the loop of the whole notebook.


## Further reading: the Ragas evaluation framework

**Ragas** (Retrieval-Augmented Generation Assessment) is an open-source evaluation framework designed for RAG systems. It quantifies both the retrieval and the generation stage without requiring human-labelled reference answers.

### Core metrics

- **Faithfulness**: whether the generated answer is supported by the supplied context, detecting whether the model invented information that is not in the context.

- **Answer Relevancy**: how relevant the generated answer is to the question, penalising answers that miss the question or carry irrelevant filler.

- **Context Relevancy**: how relevant the retrieved passages are to the question, i.e. the share of irrelevant content removed.

- **Context Recall**: whether the retrieved context contains all the information needed to answer the question.

### Why use Ragas

1. **Automated**: no human-labelled gold answers needed, so RAG pipelines can iterate quickly;
2. **End to end**: covers retrieval and generation together, making it easy to tell "recall is not enough" from "generation is off";
3. **Friendly LangChain integration**: pass LangChain `retriever` and `chain` results straight in for scoring.

### How to adopt it

In a real project, first get the evaluation loop running with the LLM-as-Judge method in this section, then migrate to Ragas step by step. The typical sequence: prepare 50+ evaluation questions -> get questions and answers from LangChain -> compute Ragas built-in metric averages -> localise the optimisation direction by splitting "retrieval metrics" from "generation metrics".

With Ragas in place, a RAG system is no longer just "able to answer" but can continuously and quantitatively "answer better".
